# 01 - Preprocessing: one notebook, either dataset

Run this notebook live (interactive figures) on Binder:
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/smehri/CATraj/main?labpath=notebooks%2F01_preprocessing.ipynb)
Or view it on [nbviewer](https://nbviewer.org/github/smehri/CATraj/blob/main/notebooks/01_preprocessing.ipynb). GitHub's own viewer does not run JavaScript, so the interactive figures do not show there.

Raw AIS files (and, for one dataset, weather, sea-state and coastline files) are written the way their providers
wrote them: coded column names, "not available" codes hidden inside the numbers, unchecked ranges, duplicated
messages, vessels that are not ships. This notebook turns them into the clean tables every other notebook reads.

**It is one notebook for both datasets.** The setting `DATASET` below (`"zenodo"` or `"us"`, or the environment
variable `CATRAJ_DATASET`) chooses which dataset it processes; the sequence of steps, the function names and the
code of every cell are the same. Where a step truly differs (the file reader, where the vessel type comes from,
the grid bearings of the US data, weather and sea state that only the Zenodo archive has) the cell says so and prints
a note for the chosen dataset.

It has three parts.

1. **The sample, cell by cell.** Every preprocessing step is run on the committed raw sample of the dataset, one step
   per cell. Before each step you find what the rule is, why it exists and where it comes from; after it, what it did
   to the data. The last cell writes the result to the dataset's `sample/preprocessed/` folder, which is committed
   and which the other notebooks read, and checks that the committed files are reproduced byte for byte.
2. **Download the full data** (off by default): each file, one at a time, with progress; nothing already in place is
   downloaded again.
3. **Preprocess the full data** (off by default): the same steps, in one call. When it is off, the report of an
   existing full run is loaded and summarised.

| | Zenodo (`DATASET = "zenodo"`) | US (`DATASET = "us"`) |
|---|---|---|
| Part 1 reads | `data/ZenodoData/sample/raw/` | `data/UsAIS/sample/raw/` |
| Part 1 writes | `data/ZenodoData/sample/preprocessed/` (committed) | `data/UsAIS/sample/preprocessed/` (committed) |
| Part 2 downloads | zenodo.org, `scripts/download_data.py` | NOAA, `scripts/download_us.py` |
| Part 2 writes | `data/ZenodoData/` (not committed) | `data/UsAIS/csv2017/`, `data/UsAIS/shoreline/` (not committed) |
| Part 3 writes | `data/ZenodoData/preprocessed/` (not committed) | `data/UsAIS/preprocessed/` (not committed) |

The rules are documented, with their sources, in the module docstrings of `src/catraj/preprocessing/archive.py`
(Zenodo) and `src/catraj/preprocessing/marinecadastre.py` (US); the notebook follows those documents.

## The two datasets

| | **Zenodo** (`"zenodo"`) | **US** (`"us"`) |
|---|---|---|
| What | AIS, weather, sea state and coastline off Brittany | AIS and shoreline of the US east coast |
| Period | 2015-10 to 2016-03, **without January 2016** | 2017-11-01 to 2017-12-31 (61 daily files) |
| Area | longitude -10 to 0, latitude 45 to 51 (the archive's own box) | the east-coast polygon (below) |
| Metric CRS | EPSG:32630, UTM 30N | EPSG:32618, UTM 18N |
| Size | 19 million AIS messages | 466 million messages, 115 million kept in the area |
| Source | Ray et al. (2019), Zenodo, [doi:10.5281/zenodo.1167595](https://zenodo.org/records/1167595) | MarineCadastre.gov / NOAA: <https://hub.marinecadastre.gov>, daily files at <https://noaaocm.blob.core.windows.net/ais/csv2/csv2017/index.html>; shoreline: NOAA Medium Resolution Shoreline |
| Licence | AIS CC BY-NC-SA 4.0 (NARI, Brest); sea state, weather and coastline public with the source cited (`data/LICENSE.md`) | public data of the US Government; cite MarineCadastre.gov and NOAA |
| Used by | notebooks **02** (annotation with weather and sea state) and **04** (CADP) | notebook **03** (2stage-CPLS) and the CLSTM prediction notebook |
| Committed sample | 2015-11-09 to 2015-11-11 (565 vessels) | 2017-11-01 and 2017-11-02, New York area |

**Zenodo.** Four kinds of files: **AIS** (`P1_AIS_Data/nari_dynamic.csv`, plus `nari_static.csv` with each ship's
type and the code tables in `P1_AIS_Codes/`), **weather** (`E2_Weather_Conditions/`, hourly observations of 16 coastal
stations), **sea state** (`E1_ocean/oc_<month>.csv`, a WAVEWATCH III forecast of depth, sea level and waves on a grid, every
3 hours) and **coastline** (`C2_European_Coastline/`, the EEA land polygons). **January 2016 is left out of every
dataset**: the archive has no sea state for it, because its `oc_january.csv` is a byte-for-byte copy of
`oc_november.csv` (the zip's MD5 equals the one Zenodo publishes, so the copy is in the source). So that every
notebook works on the same period, the AIS, weather and sea-state rows of January are dropped, and the data covers
October to December 2015 and February to March 2016. The sample is from November, so nothing is dropped from it, but
the rule is run (and shown) in Part 1 like every other.

**US.** The AIS broadcasts of the US Coast Guard, as MarineCadastre.gov publishes them, one CSV per day
(`ais-2017-MM-DD.csv.zst`), for the "eastern coast" of the United States in November and December 2017, together with
the NOAA Medium Resolution Shoreline (lines, EPSG:4269) as the barrier of the compression. This is the data of the
2021 paper ([Mehri2021] section III-B). The **study area** is the box longitude -82 to -60, latitude 23 to 46, **minus**
the Gulf side of Florida (longitude -82 to -80.9, latitude 25.2 to 31) and **minus** the Great Lakes (longitude -82 to -75,
latitude 41.3 to 46): `catraj.preprocessing.marinecadastre.AREA`. A map of it, with the sample's region and the
shoreline, is drawn in section 1.4. There is no weather and no sea state for the US data; the cells for them are skipped.

**By default the notebook runs on the committed samples** (Part 1 needs nothing else). **To process the full data**, set
`DOWNLOAD = True` (Part 2) and `PREPROCESS_FULL = True` (Part 3) below and run those sections.

In [1]:
import os

DATASET = os.environ.get("CATRAJ_DATASET", "zenodo")  # "zenodo": Brest, Zenodo archive | "us": US east coast, MarineCadastre Nov-Dec 2017
N_JOBS = int(os.environ.get("CATRAJ_N_JOBS", "-1"))   # -1: every core; 1: serial

DOWNLOAD = False         # Part 2: download the full data (Zenodo about 825 MB; US about 12.5 GB), resumable, never twice
PREPROCESS_FULL = False  # Part 3: preprocess the downloaded data (Zenodo about 2 minutes; US much longer)
FORCE_FULL = False       # Part 3: rebuild the full output even if it exists (never done unless True)

### The settings

* `DATASET` chooses the dataset for the whole notebook (`spec = get_dataset(DATASET)` below gives its paths, CRS, area,
  period and the module with the step functions). Part 1 always runs on that dataset's sample.
* `N_JOBS` is the number of processes (see below).
* `DOWNLOAD` and `PREPROCESS_FULL` are **off**: the committed notebook must run in a few minutes on a fresh clone
  without a download of several GB. The sections still show what they need: the download section reports what is
  present, and the preprocessing section loads and displays the report of an existing full run.
* `FORCE_FULL = True` rebuilds the full output (`force=True`). Nothing is ever deleted, or downloaded again, when it is
  already in place.

## Parallel processing

`N_JOBS` follows the scikit-learn convention: `1` runs serially in this process, `-1` uses every core, `k` uses
`k` worker processes (`catraj._parallel`, the standard library's `ProcessPoolExecutor`). Results come back in the
order of the tasks, so a parallel run writes **the same files and the same report** as a serial one (the package
tests compare them).

What runs in parallel in this notebook, all through the `n_jobs` argument of the package:

* **`denoise_ais(..., n_jobs=N_JOBS)`**, the 30 s median filter (the same call for both datasets). Vessels are independent, and
  `iter_denoised_ais` cuts the table into batches of whole vessels (`batch_rows`, 1.5 million rows by default) to bound memory:
  the batches are filtered on `n_jobs` processes, `n_jobs` at a time. Each sample (about 240,000 to 470,000 rows) is a single
  batch, so Part 1 runs this step in one process whatever `N_JOBS` is.
* **`spec.preprocess(..., n_jobs=N_JOBS)`**, the whole dataset. *Zenodo* (`preprocess_archive`): the datasets are independent: weather,
  each month of sea state (five on the full archive) and the coastline run side by side first; then the AIS (whose batches already use
  the processes, so no pool is started inside a pool). *US* (`preprocess_us`): the daily files of a month are read, cleaned and
  clipped to the study area side by side, then the month's vessels are filtered in batches on the processes; the two months are
  done one after the other. The steps *inside* one dataset run in order, since each works on the output of the one before.

Memory on the full data. *Zenodo* (19 million AIS rows read, 14.9 million after cleaning; estimates, not measurements): serially the
cleaned AIS table (a little over 1 GB in memory, 334 MB as Parquet) is held once and filtered one batch at a time; in parallel every
worker also holds one batch and its filtered copy, and every sea-state month in flight holds its whole table (13 million rows, about
1 GB or more). Plan for about 10 GB with all cores. *US*: a month is about 60 million positions after clipping, which is 4 to 5 GB as a
frame, and each worker holds one daily file or one batch (`scripts/prepare_us.py` reports the peak memory it measured). With less memory use
`N_JOBS = 4`, a smaller `batch_rows`, or `N_JOBS = 1`. To run serially: `N_JOBS = 1`, or the environment variable `CATRAJ_N_JOBS=1`.

**Measured on the full Zenodo archive** (a 16-core, 34 GB Windows machine, shared with other jobs while measuring, output to a scratch folder, the files
compared afterwards): `n_jobs=1` took **151 s** (97 s when the machine was quieter) and `n_jobs=-1` took **89 s** (82 s with `batch_rows=500_000`). The AIS
files and the reports of the serial and parallel runs are identical (`DataFrame.equals` and the report text). The gain is limited because reading the
19 million-row CSV and the cleaning steps before the filter (MMSI, duplicates, position jumps) run in the main process; the median filter and the
sea-state months are what is spread over cores.

In [2]:
import warnings
warnings.filterwarnings("ignore", message="Could not detect GDAL")
import catraj      # first: it sets MKL_THREADING_LAYER before NumPy loads (see catraj/__init__.py)
import hashlib
import importlib.util
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import hvplot.pandas  # noqa: F401  (registers .hvplot on DataFrames and GeoDataFrames)
import holoviews as hv
import geoviews as gv
import cartopy.crs as ccrs
from IPython.display import display, Markdown
from shapely.geometry import box

from catraj.datasets import repo_root
from catraj.preprocessing import PreprocessingReport, get_dataset, mmsi_class, mmsi_classes

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 30)
hv.extension("bokeh")          # interactive figures: zoom, pan, hover; the legend toggles a layer

spec = get_dataset(DATASET)    # paths, CRS, area, period and the step module of the chosen dataset
steps = spec.steps             # catraj.preprocessing.archive or .marinecadastre: the same function names
CRS = spec.crs                 # the metric CRS of the x and y columns
UTM = ccrs.UTM(CRS - 32600)    # EPSG:32630 -> zone 30, EPSG:32618 -> zone 18 (northern hemisphere)
PLATE = ccrs.PlateCarree()     # lon/lat

ROOT = repo_root()
RAW, OUT = spec.sample_raw, spec.sample_preprocessed      # Part 1: the committed sample
FULL_RAW, FULL_OUT = spec.raw, spec.preprocessed          # Parts 2 and 3: the full data (not committed)

rep = PreprocessingReport()          # records every step of Part 1, to be compared with the written report


def when(seconds):
    "unix seconds -> readable UTC time"
    t = pd.to_datetime(seconds, unit="s", utc=True)
    return (t.dt if isinstance(t, pd.Series) else t).strftime("%Y-%m-%d %H:%M:%S")


def show(report, since=0):
    "the report lines added since the first `since` lines"
    cols = ["step", "rule", "rows_in", "rows_out", "rows_dropped", "values_to_nan"]
    return report.to_frame().iloc[since:][cols].reset_index(drop=True)


def note(text):
    "a dataset-specific remark, shown as formatted text"
    display(Markdown(text))


def rel(path):
    return Path(path).relative_to(ROOT).as_posix()


print("dataset:", DATASET, "-", spec.title)
print("repository:", ROOT)
print("sample raw:", rel(RAW), "| sample preprocessed:", rel(OUT), "| CRS: EPSG:%d" % CRS)
print("parts the dataset has:", ", ".join(spec.has))
print("period:", when(spec.period[0]), "to", when(spec.period[1]))

dataset: us - US MarineCadastre AIS, east coast (Nov-Dec 2017)
repository: C:\dev\CATraj
sample raw: data/UsAIS/sample/raw | sample preprocessed: data/UsAIS/sample/preprocessed | CRS: EPSG:32618
parts the dataset has: ais, coastline
period: 2017-11-01 00:00:00 to 2017-12-31 23:59:59


# Part 1 - The sample, one step per cell

## 1.0 What the raw sample looks like

`sample/raw/` of the dataset has the provider's own folders, file names and layout, cut to two days (and, for the US, to
a region around New York) by `scripts/make_sample.py` or `scripts/make_us_sample.py`. The cell lists its files.

In [3]:
rows = [(rel(p).split("/sample/raw/")[1], f"{p.stat().st_size / 1e6:.2f} MB") for p in sorted(RAW.rglob("*")) if p.is_file()]
display(pd.DataFrame(rows, columns=["file", "size"]))
print("total:", f"{sum(p.stat().st_size for p in RAW.rglob('*') if p.is_file()) / 1e6:.1f} MB")

,file,size
0,ais-2017-11-01.csv.zst,3.41 MB
1,ais-2017-11-02.csv.zst,2.69 MB
2,shoreline/medium_shoreline_metadata_d.html,0.01 MB
3,shoreline/us_medium_shoreline.cpg,0.00 MB
4,shoreline/us_medium_shoreline.dbf,0.16 MB
5,shoreline/us_medium_shoreline.prj,0.00 MB
6,shoreline/us_medium_shoreline.shp,0.53 MB
7,shoreline/us_medium_shoreline.shx,0.00 MB
8,shoreline/us_medium_shoreline.xml,0.02 MB


total: 6.8 MB


Here is the first AIS file as the provider wrote it. The column names of the Zenodo archive are codes (`sourcemmsi`, `t`,
`rateofturn`, ...) and its units are those of the AIS standard; the folder's `README_parameters.txt` gives the units and the
"not available" codes, and the rules below take their limits from it. The US files (one per day, compressed with zstd)
have the MarineCadastre column names (`mmsi`, `base_date_time`, `longitude`, `sog`, ...) and no README: the rules follow
the module docstring of `marinecadastre.py`.

In [4]:
first_file = spec.ais_path(RAW)
if first_file.is_dir():
    first_file = sorted(first_file.glob("ais-*.csv*"))[0]
print("first AIS file:", rel(first_file))
if DATASET == "zenodo":
    print((first_file.parent / "README_parameters.txt").read_text(errors="replace")[:1800])
    display(pd.read_csv(first_file, nrows=4))
else:
    display(pd.read_csv(first_file, nrows=4, compression="zstd"))

first AIS file: data/UsAIS/sample/raw/ais-2017-11-01.csv.zst


,mmsi,base_date_time,longitude,latitude,sog,cog,heading,vessel_name,imo,call_sign,vessel_type,status,length,width,draft,cargo,transceiver
0,367658140,2017-11-01 00:00:01,-74.17059,40.64659,9.9,87.8,88.0,DYLAN COOPER,IMO9769934,WDH8454,NaN,0.0,36.0,11.0,NaN,NaN,NaN
1,367568460,2017-11-01 00:00:06,-73.97156,40.70313,0.0,312.5,NaN,GRAND REPUBLIC,IMO8986145,WDG7555,1012.0,NaN,93.0,16.0,NaN,NaN,NaN
2,367379570,2017-11-01 00:00:02,-74.02164,40.75228,0.0,0.0,NaN,CORNUCOPIA DESTINY,NaN,WDE6386,1012.0,NaN,29.0,10.0,NaN,NaN,NaN
3,338222451,2017-11-01 00:00:03,-73.94730,40.58332,0.0,NaN,NaN,NaN,NaN,NaN,NaN,8.0,NaN,NaN,NaN,NaN,NaN


## 1.1 AIS

### Step 1: read the files with descriptive names (`read_ais_raw`)

**What.** `read_ais_raw` reads the raw AIS and renames the coded columns to one set of descriptive names (`mmsi`,
`unix_time`, `speed_over_ground`, `course_over_ground`, `true_heading`, `rate_of_turn`, `lon`, `lat`, ...), and stores the
measurements as `float32`. *Zenodo*: one CSV, `nari_dynamic.csv`, renamed from `sourcemmsi`, `t`, `speedoverground`, ... *US*:
the daily `.csv.zst` files of the folder, renamed from `base_date_time`, `longitude`, `sog`, `cog`, ...; MarineCadastre has no rate
of turn, so that column is added empty (NaN) to keep one layout. The US reader also **clips to the study area while it reads**,
because a day holds about 8 million positions of all US waters, and counts (for information only, nothing is dropped) the
positions in the St. Lawrence / Lake Champlain part of the area.

**Why.** Descriptive names make the later rules readable, and float32 halves the memory (19 million rows for Zenodo; 235
million rows read for the US month of November).

**Source.** The names and units are those of `P1_AIS_Data/README_parameters.txt` (Zenodo) and of the MarineCadastre files
(`marinecadastre.RAW_COLUMNS`).

Every step takes an optional `report` and records the rows in and out and the values it set to NaN. We pass one
report through all of Part 1; the table under each cell shows the lines that step added.

In [5]:
n0 = len(rep)
ais = steps.read_ais_raw(spec.ais_path(RAW), rep)
display(show(rep, n0))
display(ais.head())
print(ais.dtypes.to_string())

,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,read_ais_raw,rows read,465852,465852,0,0
1,read_ais_raw,position inside the study area,465852,465852,0,0
2,read_ais_raw,"information only, kept: of these, lat > 44.5 and lon < -73 (St. Lawrence, La...",0,0,0,0


,mmsi,unix_time,lon,lat,speed_over_ground,course_over_ground,true_heading,rate_of_turn,navigational_status,ship_type_code,length,width
0,367658140,1509494401,-74.17059,40.64659,9.900000,87.800003,88.0,NaN,0.0,NaN,36.0,11.0
1,367568460,1509494406,-73.97156,40.70313,0.000000,312.500000,NaN,NaN,NaN,1012.0,93.0,16.0
2,367379570,1509494402,-74.02164,40.75228,0.000000,0.000000,NaN,NaN,NaN,1012.0,29.0,10.0
3,338222451,1509494403,-73.94730,40.58332,0.000000,NaN,NaN,NaN,8.0,NaN,NaN,NaN
4,367036190,1509494402,-74.03947,40.53952,27.700001,175.500000,177.0,NaN,0.0,NaN,NaN,NaN


mmsi                     int64
unix_time                int64
lon                    float64
lat                    float64
speed_over_ground      float32
course_over_ground     float32
true_heading           float32
rate_of_turn           float32
navigational_status    float32
ship_type_code         float32
length                 float32
width                  float32


### Step 2: which MMSI classes are there? (`mmsi_check`)

The MMSI (Maritime Mobile Service Identity) is the number that identifies a transmitting station. Not every
station is a ship, and the data contain placeholders, distress beacons and aids to navigation next to the
ships. Before dropping anything, `mmsi_check` shows what classes occur. The rule is the same for both datasets.

**The rule (ITU-R M.585).** An MMSI has **nine digits**. A **ship station** is `MIDxxxxxx`: its first three
digits are a *Maritime Identification Digit* (MID), the flag state, and the MID must be one that the ITU has
allocated (`catraj.preprocessing.ITU_MIDS`, the ITU's own table of 2022; the Zenodo archive's country list is incomplete:
it misses Malta 229, Panama 373 to 374, Singapore 566 and others). Other patterns are other kinds of station:

| pattern | class |
|---|---|
| `MIDxxxxxx` (first digit 2 to 7) | ship station |
| `00MIDxxxx` | coast station |
| `0MIDxxxxx` | group of ships |
| `111MIDxxx` | SAR aircraft |
| `99MIDxxxx` | aid to navigation (AtoN) |
| `98MIDxxxx` | craft associated with a parent ship |
| `970xxxxxx`, `972xxxxxx`, `974xxxxxx` | distress devices (SART, MOB, EPIRB) |
| `8xxxxxxxx` | handheld VHF |
| one repeated digit, `123456789`, `012345678` | placeholder (a transmitter nobody configured) |

An MMSI stored as an integer loses its leading zeros: `2275000` is really the coast station `002275000`.
Only values above 999,999,999 or at most 0 are "not nine digits". The cell classifies a few examples (with a
valid and an invalid MID for ships), then the whole sample. (The paper removed only the MMSIs that are not nine digits;
the rule here is stricter, for the US data as well.)

In [6]:
examples = [227705102, 218551000, 200123456, 123456789, 2275000, 3669991, 992271234, 982271234,
            970123456, 111227123, 812345678, 12345, 0, 1_234_567_890]
display(pd.DataFrame({"mmsi": examples, "class": [mmsi_class(m) for m in examples]}))

check = steps.mmsi_check(ais)
print("the classes found in the sample's AIS data (what the Zenodo run writes to AIS/mmsi_check.csv):")
display(check)

,mmsi,class
0,227705102,ship
1,218551000,ship
2,200123456,"ship, unallocated MID"
3,123456789,placeholder
4,2275000,coast station
5,3669991,coast station
6,992271234,aid to navigation
7,982271234,craft of a parent ship
8,970123456,distress device (SART/MOB/EPIRB)
9,111227123,SAR aircraft


the classes found in the sample's AIS data (what the Zenodo run writes to AIS/mmsi_check.csv):


,mmsi_class,MMSIs,positions
0,ship,507,465798
1,craft of a parent ship,2,54


### Step 3: keep the ship stations (`drop_invalid_mmsi`)

**What.** Rows whose MMSI is not a ship station with an allocated MID are dropped. The report has one line per class
dropped and one for the rows kept.

**Why.** A distress device or a placeholder is not a ship with a trajectory, and an identifier that is not
valid cannot be trusted to identify one vessel: nothing can be done with the row, so it is dropped (unlike a
single bad measurement, which only becomes NaN).

**Source.** ITU-R M.585 (see step 2); implementation in `catraj.preprocessing.mmsi`.

In [7]:
n0, before = len(rep), ais
ais = steps.drop_invalid_mmsi(ais, rep)
display(show(rep, n0))
gone = before.loc[before.index.difference(ais.index)].copy()
gone["mmsi_class"] = mmsi_classes(gone["mmsi"])
print("the rows that were dropped:", len(gone))
display(gone.head(10))

,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,drop_invalid_mmsi,mmsi: craft of a parent ship,54,0,54,0
1,drop_invalid_mmsi,mmsi: ship station kept,465852,465798,54,0


the rows that were dropped: 54


,mmsi,unix_time,lon,lat,speed_over_ground,course_over_ground,true_heading,rate_of_turn,navigational_status,ship_type_code,length,width,mmsi_class
395334,982470046,1509635562,-73.99851,40.76892,0.0,303.500000,NaN,NaN,NaN,NaN,NaN,NaN,craft of a parent ship
396443,982470169,1509635954,-73.99707,40.76839,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,craft of a parent ship
396764,982470046,1509636042,-73.99931,40.76872,0.2,105.099998,NaN,NaN,NaN,NaN,NaN,NaN,craft of a parent ship
397235,982470046,1509636163,-73.99916,40.76872,0.3,90.300003,NaN,NaN,NaN,NaN,NaN,NaN,craft of a parent ship
397296,982470046,1509636254,-73.99950,40.76869,1.6,325.500000,NaN,NaN,NaN,NaN,NaN,NaN,craft of a parent ship
397534,982470169,1509636343,-73.99723,40.76815,3.0,298.500000,NaN,NaN,NaN,NaN,NaN,NaN,craft of a parent ship
398069,982470169,1509636527,-73.99791,40.76854,2.2,111.300003,NaN,NaN,NaN,NaN,NaN,NaN,craft of a parent ship
398202,982470046,1509636853,-73.99965,40.76916,1.2,64.599998,NaN,NaN,NaN,NaN,NaN,NaN,craft of a parent ship
398241,982470169,1509636882,-73.99925,40.76894,3.5,101.599998,NaN,NaN,NaN,NaN,NaN,NaN,craft of a parent ship
398453,982470046,1509636943,-73.99901,40.76882,1.6,96.199997,NaN,NaN,NaN,NaN,NaN,NaN,craft of a parent ship


### Step 4: the time must lie in the dataset's period (`drop_invalid_time`)

**What.** A row whose `unix_time` is missing or outside the period (both ends included) is dropped. *Zenodo*: 2015-10-01
00:00:00 to 2016-03-31 23:59:59 UTC. *US*: 2017-11-01 00:00:00 to 2017-12-31 23:59:59 UTC. `spec.period` holds it.

**Why.** The clock of a transmitter can be wrong (a time of 1970, or of the future). Such a row cannot be placed
among the vessel's other positions.

**Source.** The archive's README, "Period" (Zenodo); [Mehri2021] section III-B (US).

In [8]:
period = spec.period
print("period:", when(period[0]), "to", when(period[1]))
print("sample's times:", when(ais.unix_time.min()), "to", when(ais.unix_time.max()))

demo = pd.DataFrame({"unix_time": [period[0] - 1, period[0], period[1], period[1] + 1, 0]})
demo["time"] = when(demo.unix_time)
demo_rep = PreprocessingReport()
kept = steps.drop_invalid_time(demo, demo_rep, period=period)
print("\nrule on five made-up times (the sample has no bad time):")
display(demo.assign(kept=demo.index.isin(kept.index)))

n0 = len(rep)
ais = steps.drop_invalid_time(ais, rep, period=period)
print("the sample:")
display(show(rep, n0))

period: 2017-11-01 00:00:00 to 2017-12-31 23:59:59
sample's times: 2017-11-01 00:00:00 to 2017-11-02 23:59:59

rule on five made-up times (the sample has no bad time):


,unix_time,time,kept
0,1509494399,2017-10-31 23:59:59,False
1,1509494400,2017-11-01 00:00:00,True
2,1514764799,2017-12-31 23:59:59,True
3,1514764800,2018-01-01 00:00:00,False
4,0,1970-01-01 00:00:00,False


the sample:


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,drop_invalid_time,unix_time inside the archive period,465798,465798,0,0


### Step 5: leave an excluded period out (`drop_excluded_periods`, Zenodo only)

**What.** Rows inside `EXCLUDED_PERIODS` are dropped. There is one period: 2016-01-01 00:00:00 to 2016-01-31
23:59:59 UTC. The same function runs on the weather and sea-state rows later. Pass `exclude=()` to
`preprocess_archive` to keep January's AIS and weather. The US data has no excluded period: the cell says so.

**Why.** The Zenodo archive has no sea state for January: `oc_january.csv` repeats November (times 2015-11-01 to
2015-11-30). To work on one period in every notebook, January is removed everywhere, and the data covers
October to December 2015 and February to March 2016 (decided with the author on 2026-11-25).

**Source.** The comparison of the archive's files (the MD5 of `[E1] Ocean Conditions.zip` equals the one Zenodo
publishes, so the copy is in the source); see step 21 for the check.

The sample is from November: nothing is dropped. The demonstration shows the rule's edges on made-up times.

In [9]:
if hasattr(steps, "drop_excluded_periods"):
    for first, last, reason in steps.EXCLUDED_PERIODS:
        print(f"excluded: {when(first)} to {when(last)}  ({reason})")

    edges = [1451606400 - 1, 1451606400, 1451606400 + 15 * 86400, 1454284799, 1454284800]
    demo = pd.DataFrame({"unix_time": edges, "time": when(np.array(edges))})
    demo_rep = PreprocessingReport()
    kept = steps.drop_excluded_periods(demo, report=demo_rep)
    print("\nrule on made-up times around January 2016:")
    display(demo.assign(kept=demo.index.isin(kept.index)))
    display(show(demo_rep))

    n0 = len(rep)
    ais = steps.drop_excluded_periods(ais, steps.EXCLUDED_PERIODS, rep)
    print("the sample (November, so nothing to drop):")
    display(show(rep, n0))
else:
    print(f'the "{DATASET}" dataset has no excluded period: nothing to do (the rule exists for the Zenodo archive only)')

the "us" dataset has no excluded period: nothing to do (the rule exists for the Zenodo archive only)


### Step 6: the position must be valid and inside the area (`drop_invalid_position`)

**What.** Two rules, in this order. (1) `lon` is in -180..180 and `lat` in -90..90; the AIS standard codes
"position not available" as **181** and **91**, which fall outside. (2) The position is inside the dataset's area
(`spec.area`). *Zenodo*: the archive's own box, longitude -10..0 and latitude 45..51. *US*: the east-coast study area (the
box minus the Gulf side of Florida and minus the Great Lakes). A row failing either is dropped. The US reader has
already clipped to the study area (step 1), so there the second rule finds nothing left to drop; the demonstration
shows that it would.

**Why.** A message without a position is useless for trajectories. A position outside the area the dataset covers is a
decoding error or a wrong fix (a ship in Brest cannot be in the Gulf of Guinea a moment later).

**Source.** The AIS standard (ITU-R M.1371) for the 181 / 91 codes; the archive's README, "Coverage" (Zenodo); the study area decided with the
author (US, see `marinecadastre.py`).

In [10]:
print("area:", spec.area)
print("sample's positions: lon", ais.lon.min(), "to", ais.lon.max(), "| lat", ais.lat.min(), "to", ais.lat.max())

lon0, lat0 = float(ais.lon.median()), float(ais.lat.median())          # a point inside the area
demo = pd.DataFrame({"lon": [lon0, 181.0, -100.0, lon0, lon0], "lat": [lat0, 91.0, 40.0, 80.0, lat0],
                     "what": ["fine (the sample's median position)", "not available (181, 91)", "far west of the area",
                              "far north of the area", "fine"]})
if DATASET == "us":
    demo = pd.concat([demo, pd.DataFrame({"lon": [-83.0, -80.0], "lat": [28.0, 43.0],
                                          "what": ["Gulf side of Florida (cut out)", "Lake Erie, Great Lakes (cut out)"]})],
                     ignore_index=True)
demo_rep = PreprocessingReport()
kept = steps.drop_invalid_position(demo, spec.area, demo_rep)
print("\nrule on made-up positions (the sample has none that fail):")
display(demo.assign(kept=demo.index.isin(kept.index)))
display(show(demo_rep))

n0 = len(rep)
ais = steps.drop_invalid_position(ais, spec.area, rep)
print("the sample:")
display(show(rep, n0))

area: StudyArea(US east coast: box -82..-60, 23..46 minus Gulf Florida, Great Lakes)
sample's positions: lon -74.32739 to -73.60017 | lat 40.2 to 40.94987

rule on made-up positions (the sample has none that fail):


,lon,lat,what,kept
0,-74.04026,40.67499,fine (the sample's median position),True
1,181.00000,91.00000,"not available (181, 91)",False
2,-100.00000,40.00000,far west of the area,False
3,-74.04026,80.00000,far north of the area,False
4,-74.04026,40.67499,fine,True
5,-83.00000,28.00000,Gulf side of Florida (cut out),False
6,-80.00000,43.00000,"Lake Erie, Great Lakes (cut out)",False


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,drop_invalid_position,"lon, lat in valid range",7,6,1,0
1,drop_invalid_position,position inside study area,6,2,4,0


the sample:


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,drop_invalid_position,"lon, lat in valid range",465798,465798,0,0
1,drop_invalid_position,position inside study area,465798,465798,0,0


### Step 7: the "not available" codes become NaN (`unavailable_to_nan`)

**What.** The AIS standard cannot leave a field empty, so it reserves a value for "not available". Here they
become NaN, and the row is kept (the other fields are fine):

| field | code | rule |
|---|---|---|
| `speed_over_ground` | 102.3 knots | 102.3 or more |
| `course_over_ground` | 360 degrees | 360 or more |
| `true_heading` | 511 degrees | 360 or more (511 is the code) |
| `rate_of_turn` | -128 | exactly -128 (Zenodo only; empty for the US) |

**Why.** Left as numbers, a heading of 511 degrees or a speed of 102.3 knots would be filtered, averaged and
compressed as if real. NaN is ignored by the median filter (and stays NaN: nothing is imputed). The 2017 US files already
carry NaN instead of the codes (the largest values found are 96.6 knots, 359.9 and 359 degrees); the rule stays as a guard
and the report counts what it finds (nothing).

**Source.** `README_parameters.txt` of `P1_AIS_Data` and ITU-R M.1371.

In [11]:
cols = ["speed_over_ground", "course_over_ground", "true_heading", "rate_of_turn"]
codes = {"speed_over_ground": ais.speed_over_ground >= 102.3, "course_over_ground": ais.course_over_ground >= 360,
         "true_heading": ais.true_heading >= 360, "rate_of_turn": ais.rate_of_turn == -128}
nan_before = ais[cols].isna().sum()
print("heading values equal to 511 before:", int((ais.true_heading == 511).sum()))

n0 = len(rep)
ais = steps.unavailable_to_nan(ais, rep)
display(show(rep, n0))

display(pd.DataFrame({"rows with the code": {c: int(m.sum()) for c, m in codes.items()},
                      "NaN before": nan_before, "NaN after": ais[cols].isna().sum()}))
print("true_heading is NaN for", f"{ais.true_heading.isna().mean():.1%}", "of the sample: many vessels send no heading")
if DATASET == "us":
    print("rate_of_turn is NaN for", f"{ais.rate_of_turn.isna().mean():.0%}", "of the rows: MarineCadastre has no rate of turn")

heading values equal to 511 before: 0


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,unavailable_to_nan,speed_over_ground >= 102.3 (n/a),465798,465798,0,0
1,unavailable_to_nan,course_over_ground >= 360 (n/a),465798,465798,0,0
2,unavailable_to_nan,true_heading >= 360 (511 n/a),465798,465798,0,0
3,unavailable_to_nan,rate_of_turn == -128 (n/a),465798,465798,0,0


,rows with the code,NaN before,NaN after
speed_over_ground,0,146,146
course_over_ground,0,27430,27430
true_heading,0,234254,234254
rate_of_turn,0,465798,465798


true_heading is NaN for 50.3% of the sample: many vessels send no heading
rate_of_turn is NaN for 100% of the rows: MarineCadastre has no rate of turn


### Step 8: every numeric field in its range (`check_ranges`)

**What.** A value outside its range becomes NaN (the row is kept). The ranges:

| field | valid | note |
|---|---|---|
| `speed_over_ground` | 0 <= v < 102.3 knots | 102.3 is "not available" (step 7) |
| `course_over_ground` | 0 <= v < 360 degrees | |
| `true_heading` | 0 <= v < 360 degrees | |
| `rate_of_turn` | -127 to 127 | the **AIS code**, not degrees per minute (Zenodo) |
| `navigational_status` | the integers 0 to 15 | `Navigational Status.csv` |
| `length`, `width` | 0 < length <= 1022 m, 0 < width <= 126 m | the AIS fields (US only: the Zenodo static messages are read in step 11) |

`rate_of_turn` is the AIS code `4.733 * sqrt(rate)`, which covers 0 to 127 (127 * 127 / 4.733^2 = 720 degrees per
minute: that is where the Zenodo README's "0 to 720 degrees per minute" comes from). -127 and 127 mean "turning at more
than 5 degrees per 30 s, no turn indicator" and are kept. After the check the status is an `int8` column when no
value is missing.

**Why.** An out-of-range value is a decoding error. It must not reach the filter or the models, but the row's
position may still be right, so only the value goes.

**Source.** Each Zenodo folder's README, ITU-R M.1371, `P1_AIS_Codes/Navigational Status.csv`; the AIS field sizes (US).

The samples are clean, so the demonstration uses made-up values. For the US data the empty `rate_of_turn` column is dropped
after this step (the output has none), as the daily workers of `preprocess_us` do.

In [12]:
demo = pd.DataFrame({"speed_over_ground": [12.0, -1.0, 150.0, 0.0],
                     "course_over_ground": [90.0, 400.0, -5.0, 359.9],
                     "true_heading": [90.0, 360.0, 511.0, 0.0],
                     "rate_of_turn": [0.0, 200.0, -127.0, -128.0],
                     "navigational_status": [0.0, 20.0, 15.0, 3.5]})
checked = ["speed_over_ground", "course_over_ground", "true_heading", "rate_of_turn", "navigational_status"]
if DATASET == "us":
    demo["length"], demo["width"] = [100.0, -5.0, 2000.0, 50.0], [20.0, 10.0, 30.0, 200.0]
    checked += ["length", "width"]
demo_rep = PreprocessingReport()
out = steps.check_ranges(demo, demo_rep)
print("made-up values, before (top) and after (bottom):")
display(demo)
display(out)
display(show(demo_rep).query("values_to_nan > 0"))

n0 = len(rep)
ais = steps.check_ranges(ais, rep)
print("the sample:")
display(show(rep, n0))
display(ais[checked].describe().loc[["count", "min", "max"]])
print("navigational_status dtype:", ais.navigational_status.dtype,
      f"(float32 because {int(ais.navigational_status.isna().sum()):,} rows have no status: it is empty in the raw data, "
      "most likely in the class B messages, ITU 18 and 19, which carry neither status nor rate of turn; "
      "it would be int8 if none were missing)")
if DATASET == "us":
    ais = ais.drop(columns="rate_of_turn")     # all NaN: MarineCadastre has none, and the US output has no such column

made-up values, before (top) and after (bottom):


,speed_over_ground,course_over_ground,true_heading,rate_of_turn,navigational_status,length,width
0,12.0,90.0,90.0,0.0,0.0,100.0,20.0
1,-1.0,400.0,360.0,200.0,20.0,-5.0,10.0
2,150.0,-5.0,511.0,-127.0,15.0,2000.0,30.0
3,0.0,359.9,0.0,-128.0,3.5,50.0,200.0


,speed_over_ground,course_over_ground,true_heading,rate_of_turn,navigational_status,length,width
0,12.0,90.0,90.0,0.0,0.0,100.0,20.0
1,NaN,NaN,NaN,NaN,NaN,NaN,10.0
2,NaN,NaN,NaN,-127.0,15.0,NaN,30.0
3,0.0,359.9,0.0,NaN,NaN,50.0,NaN


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,check_ranges,speed_over_ground outside 0..<102.3,4,4,0,2
1,check_ranges,course_over_ground outside 0..<360,4,4,0,2
2,check_ranges,true_heading outside 0..<360,4,4,0,2
3,check_ranges,rate_of_turn outside -127..127,4,4,0,2
4,check_ranges,navigational_status not a code 0..15,4,4,0,2
5,check_ranges,length outside 0..1022,4,4,0,2
6,check_ranges,width outside 0..126,4,4,0,1


the sample:


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,check_ranges,speed_over_ground outside 0..<102.3,465798,465798,0,0
1,check_ranges,course_over_ground outside 0..<360,465798,465798,0,0
2,check_ranges,true_heading outside 0..<360,465798,465798,0,0
3,check_ranges,rate_of_turn outside -127..127,465798,465798,0,0
4,check_ranges,navigational_status not a code 0..15,465798,465798,0,0
5,check_ranges,length outside 0..1022,465798,465798,0,0
6,check_ranges,width outside 0..126,465798,465798,0,0


,speed_over_ground,course_over_ground,true_heading,rate_of_turn,navigational_status,length,width
count,465652.000000,438368.000000,231544.0,0.0,373918.0,388459.0,343301.0
min,0.000000,0.000000,0.0,NaN,0.0,5.0,2.0
max,48.599998,359.899994,359.0,NaN,12.0,431.0,48.0


navigational_status dtype: float32 (float32 because 91,880 rows have no status: it is empty in the raw data, most likely in the class B messages, ITU 18 and 19, which carry neither status nor rate of turn; it would be int8 if none were missing)


### Step 9: duplicates and order (`drop_duplicates_and_sort`)

**What.** A vessel cannot be in two places at one second: of rows with the same `(mmsi, unix_time)` only the first
is kept, and the table is sorted by `(mmsi, unix_time)` with a fresh index.

**Why.** The Zenodo archive was assembled from several receivers, and MarineCadastre merges the Coast Guard's, so the same message
arrives more than once. The filter, the position-jump rule and the trajectory methods all need each vessel's positions in time order.

**Source.** The structure of the data (one message per vessel and second).

In [13]:
key = ["mmsi", "unix_time"]
dups = ais[ais.duplicated(key, keep=False)].sort_values(key)
print(f"{len(dups):,} rows belong to a repeated (mmsi, unix_time); the first two groups:")
display(dups.head(6))

n0 = len(rep)
ais = steps.drop_duplicates_and_sort(ais, rep)
display(show(rep, n0))
print("sorted by (mmsi, unix_time):", bool((ais[key].apply(tuple, axis=1).is_monotonic_increasing)))

4 rows belong to a repeated (mmsi, unix_time); the first two groups:


,mmsi,unix_time,lon,lat,speed_over_ground,course_over_ground,true_heading,navigational_status,ship_type_code,length,width
213299,367006570,1509580799,-73.95021,40.76469,8.8,35.200001,35.0,0.0,NaN,34.0,11.0
255715,367006570,1509580799,-73.95021,40.76469,8.8,35.200001,35.0,0.0,NaN,34.0,11.0
448439,367314480,1509667170,-73.76542,40.81306,0.0,341.899994,NaN,5.0,NaN,18.0,7.0
450817,367314480,1509667170,-73.76542,40.81306,0.0,341.899994,NaN,5.0,NaN,18.0,7.0


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,drop_duplicates_and_sort,"duplicate (mmsi, unix_time)",465798,465796,2,0


sorted by (mmsi, unix_time): True


### Step 10: isolated position jumps (`drop_position_jumps`)

**What.** An isolated spike is dropped: a position whose speed from the vessel's previous position **and** to its
next position both exceed 50 knots, while the speed from the previous directly to the next does not. (If a
vessel really moves faster than the limit, every position agrees with its neighbours and none is dropped.) The first
and last position of a vessel have one neighbour: they go when the speed to it exceeds the limit and the
neighbour's own next segment does not. Speeds are great-circle distances over time; a zero time gap with a different
position is infinite speed, with the same position zero. It is one vectorised pass over all vessels, on the
positions as given, so a run of several consecutive spikes is not removed; isolated ones, the usual decoding
error, are. `max_speed_knots=None` disables the rule. The function is the same for both datasets.

**Why.** A single wrong fix, hundreds of kilometres away, would stretch the track and make the median filter,
speed estimates and compression meaningless around it. 50 knots is faster than any ship in the data except a few
wing-in-ground and military craft, and far below what a decoding error implies.

**Source.** Decided with the author; the speed limit is a parameter of `preprocess_archive` and `preprocess_us` (`max_speed_knots`).

Below: the speed between consecutive positions around each row the rule removed from the sample (the dropped row
is marked), then the rule on a made-up track.

**Order.** Zenodo runs the rule before the vessel types (step 11), as here. `preprocess_us` attaches the types first
(the most frequent code of a vessel is counted on the daily files, before duplicates and jumps are dropped); the
difference matters only if a dropped row would change a vessel's most frequent type, and the check in 1.5 shows that it does
not change the files of the sample.

In [14]:
from catraj.preprocessing.archive import _knots      # great-circle speed in knots (the rule's own function)

def with_speeds(frame):
    out = frame.copy()
    out["knots_from_previous"] = np.r_[np.nan, _knots(out.lat.to_numpy()[:-1], out.lon.to_numpy()[:-1],
                                                      out.lat.to_numpy()[1:], out.lon.to_numpy()[1:],
                                                      np.diff(out.unix_time.to_numpy(dtype=float)))]
    return out

n0, before = len(rep), ais
ais = steps.drop_position_jumps(ais, 50, rep)
display(show(rep, n0))

keys_before = pd.MultiIndex.from_frame(before[key])
removed = keys_before.difference(pd.MultiIndex.from_frame(ais[key]))
for i in np.flatnonzero(keys_before.isin(removed))[:5]:
    window = with_speeds(before.iloc[max(i - 2, 0): i + 3][["mmsi", "unix_time", "lon", "lat", "speed_over_ground"]])
    window["dropped"] = np.where(window.index == i, "<-- dropped", "")
    window["time"] = when(window.unix_time)
    display(window.round(4))

,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,drop_position_jumps,isolated jump above 50 knots,465796,465795,1,0


,mmsi,unix_time,lon,lat,speed_over_ground,knots_from_previous,dropped,time
23475,319486000,1509640528,-73.9864,40.7861,0.1,NaN,,2017-11-02 16:35:28
23476,319486000,1509640881,-74.0112,40.7629,1.0,18.2793,,2017-11-02 16:41:21
23477,319486000,1509640970,-73.9872,40.7852,0.2,69.9037,<-- dropped,2017-11-02 16:42:50
23478,319486000,1509641032,-74.0171,40.7628,2.1,110.8860,,2017-11-02 16:43:52
23479,319486000,1509641211,-73.9862,40.7866,0.0,40.2171,,2017-11-02 16:46:51


In [15]:
# the rule on a made-up track: one position 1.4 degrees of latitude off, then a vessel that really is fast
t0 = spec.period[0] + 86400 * 40
track = pd.DataFrame({"mmsi": 227705102, "unix_time": t0 + 60 * np.arange(7),
                      "lon": lon0 + 0.001 * np.arange(7), "lat": lat0})
track.loc[3, "lat"] = lat0 + 1.4                                                    # the spike
fast = pd.DataFrame({"mmsi": 227705103, "unix_time": t0 + 60 * np.arange(7),
                     "lon": lon0 + 0.03 * np.arange(7), "lat": lat0})               # 65 to 85 knots, steady
both = pd.concat([track, fast], ignore_index=True)
kept = steps.drop_position_jumps(both, 50)
both["kept"] = pd.MultiIndex.from_frame(both[key]).isin(pd.MultiIndex.from_frame(kept[key]))
display(with_speeds(both).round(3))

,mmsi,unix_time,lon,lat,kept,knots_from_previous
0,227705102,1512950400,-74.040,40.675,True,NaN
1,227705102,1512950460,-74.039,40.675,True,2.732
2,227705102,1512950520,-74.038,40.675,True,2.732
3,227705102,1512950580,-74.037,42.075,False,5043.406
4,227705102,1512950640,-74.036,40.675,True,5043.406
5,227705102,1512950700,-74.035,40.675,True,2.732
6,227705102,1512950760,-74.034,40.675,True,2.732
7,227705103,1512950400,-74.040,40.675,True,inf
8,227705103,1512950460,-74.010,40.675,True,81.965
9,227705103,1512950520,-73.980,40.675,True,81.965


### Step 11: vessel type (`spec.attach_vessel_types`)

**What.** `ship_type_code` and `vessel_type` are added to every position; the US data also gets `vessel_group`, `length`
and `width`. This is where the two datasets truly differ, and `spec.attach_vessel_types(ais, raw_root, report)` hides the difference.
*Zenodo*: the code is the one in the vessel's last static message (`nari_static.csv`, field `shiptype`) that has a type; the name is the part
before ` - ` in `P1_AIS_Codes/Ship Types List.csv` for the range the code falls in. A vessel with no static message, or a code that the
table does not list (0, "not available", for one), is `"Unknown"`; `ship_type_code` is NaN for the former. The report counts both kinds
of Unknown as two rules (in the `values_to_nan` column, the positions affected). *US*: the AIS position files carry the type code (0 to 99,
ITU-R M.1371, or one of the AVIS service codes 1001 to 1025 MarineCadastre adds), the length and the width; per vessel the most frequent
non-null value is taken (the smallest on a tie), and the names follow the official table "AIS Vessel Type and Group Codes used by the
Marine Cadastre Project" (the Zenodo names where there is one: Cargo, Tanker, Passenger, Fishing, Tug, ...). `vessel_group` is the paper's
grouping of Fig. 2: Tanker, Tug, Cargo, Sailing, Passenger, Fishing, Military, Other. A vessel with no code is `Other`.

**Why.** The type is the vessel's category (cargo, tanker, fishing, passenger, ...): later notebooks colour and select
by it. It is a property of the vessel, so it is attached to every position.

**Source.** `P1_AIS_Codes/Ship Types List.csv`, ITU-R M.1371 (ship and cargo type) for Zenodo; `VesselTypeCodes2018.pdf` of
MarineCadastre (`coast.noaa.gov/data/marinecadastre/ais/`) for the US.

In [16]:
if DATASET == "zenodo":
    static_csv = RAW / "P1_AIS_Data" / "nari_static.csv"
    types_csv = RAW / "P1_AIS_Codes" / "Ship Types List.csv"
    by_mmsi, names = steps.vessel_types(static_csv, types_csv)
    print(f"{len(by_mmsi)} vessels have a type in the static messages. Table of ship types:")
    display(pd.read_csv(types_csv))
else:
    codes_seen = ais.ship_type_code.value_counts().sort_index()
    print(f"{ais.mmsi[ais.ship_type_code.notna()].nunique()} of {ais.mmsi.nunique()} vessels broadcast a ship-type code; "
          f"the codes found in the sample, with their names (marinecadastre.VESSEL_TYPE_NAMES):")
    display(pd.DataFrame({"positions": codes_seen,
                          "name": [steps.vessel_type_name(c) for c in codes_seen.index],
                          "group": [steps.vessel_group_of(c) for c in codes_seen.index]}).head(25))

n0 = len(rep)
ais = spec.attach_vessel_types(ais, RAW, rep)
display(show(rep, n0))
display(ais.groupby("vessel_type", observed=True).agg(vessels=("mmsi", "nunique"), positions=("mmsi", "size"))
        .sort_values("positions", ascending=False))
if DATASET == "zenodo":
    in_static = ais.mmsi.isin(by_mmsi.index)
    print("vessels without a static message:", ais.loc[~in_static, "mmsi"].nunique(),
          "| vessels whose code is not in the table:", ais.loc[in_static & ais.vessel_type.eq("Unknown"), "mmsi"].nunique())
else:
    display(ais.groupby("vessel_group", observed=True).agg(vessels=("mmsi", "nunique"), positions=("mmsi", "size"))
            .sort_values("positions", ascending=False))
    print("vessels without a ship-type code (type and group Other/Unknown):", ais.loc[ais.ship_type_code.isna(), "mmsi"].nunique(),
          "| length and width are NaN for", ais.loc[ais.length.isna(), "mmsi"].nunique(), "and", ais.loc[ais.width.isna(), "mmsi"].nunique(), "vessels")

297 of 507 vessels broadcast a ship-type code; the codes found in the sample, with their names (marinecadastre.VESSEL_TYPE_NAMES):


,positions,name,group
ship_type_code,,,
30.0,2224,Fishing,Fishing
31.0,6771,Towing,Tug
32.0,2274,Towing,Tug
52.0,5310,Tug,Tug
73.0,498,Cargo,Cargo
1001.0,2113,Fishing,Fishing
1004.0,22801,Cargo,Cargo
1005.0,16984,Industrial Vessel,Other
1011.0,3205,Oil Recovery,Other


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,attach_vessel_types,Unknown / Other: vessel without a ship-type code,465795,465795,0,234171


,vessels,positions
vessel_type,,
Unknown,210,234171
Passenger,112,107272
Public Vessel,31,27727
Cargo,35,23299
Pleasure Craft,75,23243
Industrial Vessel,16,16984
Towing,5,9045
Tanker,10,7802
Tug,3,5310


,vessels,positions
vessel_group,,
Other,262,285487
Passenger,112,107272
Cargo,35,23299
Sailing,75,23243
Tug,8,14355
Tanker,10,7802
Fishing,5,4337


vessels without a ship-type code (type and group Other/Unknown): 210 | length and width are NaN for 106 and 187 vessels


### Step 12: project to metres (`project`)

**What.** `x` and `y` are added: the longitude and latitude projected to the dataset's metric CRS, `spec.crs`. *Zenodo*: **EPSG:32630**
(UTM zone 30N), the projected CRS of the whole archive area (it lies between 10 W and 0, mostly in zone 30). *US*: **EPSG:32618** (UTM
zone 18N, central meridian 75 W). The US area spans zones 17 to 19, so a position far from the central meridian is in a distorted but still
metric system, and **grid north is not true north** there (see step 13).

**Why.** The median filter, distances, the compression tolerances (metres) and the coastline all work in
metres, not degrees (a degree of longitude is not a degree of latitude in length). One CRS per dataset keeps the whole
area in one coordinate system.

**Source.** EPSG:32630 and EPSG:32618; `pyproj`.

In [17]:
from pyproj import Transformer

ais = steps.project(ais, crs=CRS)
display(ais[["lon", "lat", "x", "y"]].head(3))
back = Transformer.from_crs(CRS, 4326, always_xy=True).transform(ais.x.to_numpy(), ais.y.to_numpy())
print(f"CRS: EPSG:{CRS}")
print("round trip (projected back to lon/lat), largest error in degrees:",
      float(max(np.abs(back[0] - ais.lon).max(), np.abs(back[1] - ais.lat).max())))
print("x from", round(ais.x.min()), "to", round(ais.x.max()), "m; y from", round(ais.y.min()), "to", round(ais.y.max()), "m")

,lon,lat,x,y
0,-73.88309,40.47487,594676.353505,4.481064e+06
1,-73.87753,40.47270,595150.727316,4.480829e+06
2,-73.87233,40.47066,595594.428651,4.480609e+06


CRS: EPSG:32618
round trip (projected back to lon/lat), largest error in degrees: 2.1316282072803006e-14
x from 556998 to 618990 m; y from 4450502 to 4534104 m


### Step 13: the 30 s median filter (`denoise_ais`)

**What.** Per vessel, a centred median over a **30 second** window replaces `x`, `y`, `speed_over_ground`,
`course_over_ground` and `true_heading`. The window is counted in time, not in points (AIS messages arrive anywhere
from every 2 s to every 6 min), so a gap longer than 15 s keeps the points on either side apart. Course and heading are
angles: they are filtered on the circle (through their unit vector), so 359 and 1 degrees do not average to 180.
NaN values are ignored inside a window and stay NaN; codes (status, rate of turn) are never filtered. `lon` and
`lat` are recomputed from the filtered `x`, `y`. The rows are not dropped or reordered. The call is the same for both
datasets. *US only*: the filter also adds `course_over_ground_grid` and `true_heading_grid`, the course and heading **on the map** (grid
north): `true - meridian convergence`, with the convergence taken at the filtered position (`marinecadastre.grid_bearing`, pyproj). The
curvilinear model of the CLSTM takes grid north as north, so the course the network is given must be the course on the map. The cell shows
how far the two differ.

**Why.** Raw AIS positions are noisy. [Mehri2021] section III-B applies a moving average to the raw data before
compression; the package uses the median (a single spike does not move it, a mean smears it over every
neighbour), with a 30 s window decided with the author. The filter belongs before compression: filter, then
compress. The later notebooks therefore call the package functions with `filter_window=None` on this data:
it is already filtered, and filtering twice would change it again.

**Source.** [Mehri2021] section III-B; `catraj.preprocessing.filtering`.

`n_jobs=N_JOBS` spreads batches of vessels over processes; the sample is one batch (see "Parallel processing").
The cell then compares one busy vessel before and after.

In [18]:
n0, before = len(rep), ais.reset_index(drop=True)
gaps = before.groupby("mmsi").unix_time.diff().dropna()
print(f"time between a vessel's consecutive messages: median {gaps.median():.0f} s, 10th percentile {gaps.quantile(.1):.0f} s, "
      f"90th percentile {gaps.quantile(.9):.0f} s ({np.mean(gaps > 15):.0%} of the gaps are longer than the 15 s half-window)")
t = time.perf_counter()
ais = steps.denoise_ais(ais, 30, rep, n_jobs=N_JOBS)
print(f"filtered in {time.perf_counter() - t:.1f} s (n_jobs={N_JOBS})")
display(show(rep, n0))

shift = np.hypot(ais.x - before.x, ais.y - before.y)
print(f"position moved by the filter: median {shift.median():.2f} m, 99th percentile {shift.quantile(.99):.1f} m, "
      f"largest {shift.max():.0f} m; {np.mean(shift > 0):.1%} of the positions changed")
print(f"speed values changed: {np.mean(ais.speed_over_ground.ne(before.speed_over_ground) & before.speed_over_ground.notna()):.1%}")
print("NaN counts unchanged:", bool((ais[['speed_over_ground', 'course_over_ground', 'true_heading']].isna().sum()
                                     == before[['speed_over_ground', 'course_over_ground', 'true_heading']].isna().sum()).all()))
if DATASET == "us":
    print("The US messages are already thinned to about one per minute, so a 30 s window usually holds a single position and the filter changes "
          "next to nothing; what it does here is the projection, the unit-circle handling of the angles and the grid bearings. The filter matters for "
          "dense data such as the Zenodo archive.")

if DATASET == "us":
    ais = ais[steps.OUTPUT_COLUMNS]          # the columns, in the order, that preprocess_us writes
    gap = (ais.course_over_ground - ais.course_over_ground_grid + 180) % 360 - 180      # true minus grid course
    print("\ngrid north: course_over_ground - course_over_ground_grid (the meridian convergence), degrees:")
    display(gap.describe().round(3).to_frame("difference"))
    print("the difference grows with the distance from the central meridian (75 W): the sample's median longitude is",
          round(float(ais.lon.median()), 2), "and the convergence",
          round(float(gap.median()), 2), "degrees; at 60 W it reaches about 10 degrees.")

time between a vessel's consecutive messages: median 70 s, 10th percentile 61 s, 90th percentile 180 s (100% of the gaps are longer than the 15 s half-window)


filtered in 0.8 s (n_jobs=-1)


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,denoise_ais,"median filter, 30 s window",465795,465795,0,0


position moved by the filter: median 0.00 m, 99th percentile 0.0 m, largest 35 m; 0.0% of the positions changed
speed values changed: 0.0%
NaN counts unchanged: True
The US messages are already thinned to about one per minute, so a 30 s window usually holds a single position and the filter changes next to nothing; what it does here is the projection, the unit-circle handling of the angles and the grid bearings. The filter matters for dense data such as the Zenodo archive.

grid north: course_over_ground - course_over_ground_grid (the meridian convergence), degrees:


,difference
count,438365.000
mean,0.626
std,0.073
min,0.437
25%,0.568
50%,0.626
75%,0.651
max,0.917


the difference grows with the distance from the central meridian (75 W): the sample's median longitude is -74.04 and the convergence 0.63 degrees; at 60 W it reaches about 10 degrees.


In [19]:
# one busy, moving vessel: 300 positions from where it starts to move
moving = before[before.speed_over_ground > 3].groupby("mmsi").size()
vessel = moving.idxmax()
first = before.index[(before.mmsi == vessel) & (before.speed_over_ground > 3)][0]      # where it starts to move
sel = before.index[before.mmsi == vessel]
sel = sel[sel >= first][:300]
b, a = before.loc[sel], ais.loc[sel]
minutes = (b.unix_time - b.unix_time.iloc[0]) / 60

raw_t = pd.DataFrame({"minutes": minutes, "speed (knots)": b.speed_over_ground, "series": "raw"})
med_t = pd.DataFrame({"minutes": minutes, "speed (knots)": a.speed_over_ground, "series": "median, 30 s"})
speed = pd.concat([raw_t, med_t]).hvplot.line(
    x="minutes", y="speed (knots)", by="series", color=["#1f77b4", "#ff7f0e"], line_width=1.5,
    title=f"vessel {vessel}: speed", width=480, height=380, legend="top_right", hover_cols=["series"])

track_b = b.assign(series="raw")
track_a = a.assign(series="median, 30 s")
def path(d, color, width, label):
    return gv.Path([d[["x", "y"]].to_numpy()], crs=UTM, label=label).opts(color=color, line_width=width)
track = (path(track_b, "#1f77b4", 1, "raw")
         * track_b.hvplot.points(x="x", y="y", geo=True, crs=UTM, color="#1f77b4", size=4, label="raw", hover_cols=["unix_time"])
         * path(track_a, "#ff7f0e", 2.5, "median, 30 s"))
track = (gv.tile_sources.CartoLight * track).opts(legend_position="top_right", title="track", width=480, height=380, active_tools=["wheel_zoom"])
(speed + track).cols(2)

:Layout
   .NdOverlay.I :NdOverlay   [series]
      :Curve   [minutes]   (speed (knots),series)
   .Overlay.I   :Overlay
      .WMTS.I                 :WMTS   [Longitude,Latitude]
      .Path.Raw               :Path   [Longitude,Latitude]
      .Points.Raw             :Points   [x,y]   (unix_time)
      .Path.Median_comma_30_s :Path   [Longitude,Latitude]

In [20]:
print(f"AIS after all steps: {len(ais):,} positions, {ais.mmsi.nunique()} vessels")
print(ais.dtypes.to_string())
display(ais.head(3))

AIS after all steps: 465,795 positions, 507 vessels
mmsi                          int64
unix_time                     int64
x                           float64
y                           float64
lon                         float64
lat                         float64
speed_over_ground           float32
course_over_ground          float32
true_heading                float32
course_over_ground_grid     float32
true_heading_grid           float32
navigational_status         float32
ship_type_code              float32
vessel_type                category
vessel_group               category
length                      float32
width                       float32


,mmsi,unix_time,x,y,lon,lat,speed_over_ground,course_over_ground,true_heading,course_over_ground_grid,true_heading_grid,navigational_status,ship_type_code,vessel_type,vessel_group,length,width
0,220206000,1509494400,594676.353505,4.481064e+06,-73.88309,40.47487,15.7,117.500000,118.0,116.774940,117.274940,0.0,1004.0,Cargo,Cargo,353.0,43.0
1,220206000,1509494469,595150.727316,4.480829e+06,-73.87753,40.47270,15.2,116.900002,118.0,116.171371,117.271370,0.0,1004.0,Cargo,Cargo,353.0,43.0
2,220206000,1509494534,595594.428651,4.480609e+06,-73.87233,40.47066,14.5,116.699997,119.0,115.968018,118.268021,0.0,1004.0,Cargo,Cargo,353.0,43.0


## 1.2 Weather (datasets with weather: Zenodo)

Hourly observations at 16 coastal stations (rp5.ua, from Met Office / NOAA data), in `table_wheatherObservation.csv`
(sic: the archive's file name has the typo), with the stations in `table_weatherStation.csv` and the wind-direction
codes in `table_windDirection.csv`. **The US dataset has no weather**: for `DATASET = "us"` the cells of this section and of
1.3 print that they are skipped.

### Step 14: read the three tables (`read_weather_raw`)

The observations keep the archive's column names (`T`, `Tn`, `P`, `U`, `Ff`, ...) until the stations are merged in step 16.

In [21]:
if "weather" in spec.has:
    n0 = len(rep)
    obs, stations, wind = steps.read_weather_raw(RAW / "E2_Weather_Conditions", rep)
    display(show(rep, n0))
    display(obs.head(3))
    display(stations.head(3))
    print(len(stations), "stations;", len(wind), "wind direction codes")
else:
    print(f"no weather in the {DATASET} dataset: this step is skipped")

no weather in the us dataset: this step is skipped


### Step 15: the "missing" code -65536 becomes NaN (`weather_unavailable_to_nan`)

**What.** In every column the archive's code for "missing" is **-65536**: here it becomes NaN.

**Why.** -65536 is a number to a mean, a filter or a plot (it would be an air pressure of -65536 mm Hg). Most stations
do not report every field every hour (gusts, precipitation and the minimum and maximum temperature are mostly missing).

**Source.** `E2_Weather_Conditions/README_parameters.txt`.

In [22]:
if "weather" in spec.has:
    n0 = len(rep)
    cols = [c for c in obs.columns if c not in ("id_station", "local_time")]
    count = (obs[cols] == -65536).sum()
    obs = steps.weather_unavailable_to_nan(obs, rep)
    display(pd.DataFrame({"rows with -65536": count, "NaN after": obs[cols].isna().sum()}).T)
    display(show(rep, n0).query("values_to_nan > 0"))
else:
    print(f"no weather in the {DATASET} dataset: this step is skipped")

no weather in the us dataset: this step is skipped


### Step 16: add each station's position (`merge_weather_stations`)

**What.** The columns get descriptive names (`T` to `air_temperature`, `Ff` to `wind_speed`, `local_time` to
`unix_time`, ...) and each observation gets its station's `latitude` and `longitude`. An observation whose
station is not in the station table is dropped.

**Why.** The position is what lets an AIS point be matched to the nearest station later (notebook 02).
A station without a position cannot be used.

**Source.** `table_weatherStation.csv`, `README_parameters.txt` for the names and units.

In [23]:
if "weather" in spec.has:
    n0 = len(rep)
    obs = steps.merge_weather_stations(obs, stations, rep)
    display(show(rep, n0))
    display(obs.head(3))
    print(list(obs.columns))
else:
    print(f"no weather in the {DATASET} dataset: this step is skipped")

no weather in the us dataset: this step is skipped


### Step 17: time inside the period, January out (`drop_invalid_weather`, `drop_excluded_periods`)

The same two rules as for AIS (steps 4 and 5) on the weather's `unix_time`: inside 2015-10-01 to 2016-03-31, and
not in January 2016 (the weather of that month is left out so that all datasets cover the same period).

In [24]:
if "weather" in spec.has:
    n0 = len(rep)
    obs = steps.drop_invalid_weather(obs, rep)
    obs = steps.drop_excluded_periods(obs, steps.EXCLUDED_PERIODS, rep, dataset="weather")
    display(show(rep, n0))
    print("times:", when(obs.unix_time.min()), "to", when(obs.unix_time.max()))
else:
    print(f"no weather in the {DATASET} dataset: this step is skipped")

no weather in the us dataset: this step is skipped


### Step 18: physical ranges (`check_weather_ranges`)

**What.** A value outside its range becomes NaN. The README gives units but no limits, so the limits are physical ones, wide enough to hold every
real observation (world records, in the README's units):

| field | range |
|---|---|
| air, minimum, maximum temperature, dew point | -90 to 60 degrees C |
| `air_pressure` | 640 to 820 mm of mercury (870 to 1085 hPa is the record range) |
| `relative_humidity` | 0 to 100 % |
| `wind_speed`, the two gusts | 0 to 120 m/s |
| `visibility` | 0 to 100 km |
| `precipitation` | 0 to 1000 mm; its period 0 to 24 hours |
| `wind_direction_code` | must be in `table_windDirection.csv` |

**Why.** To catch decoding and transcription errors (a pressure of 2000, a humidity of 150) without
discarding real extremes. **Source.** World records; the units are the README's.

The sample's data are clean, so a made-up table shows the rule.

In [25]:
if "weather" in spec.has:
    demo = pd.DataFrame({"air_temperature": [12.0, 95.0], "air_pressure": [765.0, 2000.0], "relative_humidity": [80.0, 150.0],
                         "wind_speed": [5.0, 130.0], "visibility": [10.0, -3.0], "wind_direction_code": [10, 99]})
    demo_rep = PreprocessingReport()
    out = steps.check_weather_ranges(demo, wind, demo_rep)
    print("made-up values, before and after (the second row is all wrong):")
    display(demo)
    display(out)

    n0 = len(rep)
    obs = steps.check_weather_ranges(obs, wind, rep)
    print("the sample:")
    display(show(rep, n0))
    num = [c for c in obs.columns if c in steps.WEATHER_FILTERED or c == "precipitation_period"]
    display(obs[num].describe().loc[["count", "min", "max"]].round(1))
else:
    print(f"no weather in the {DATASET} dataset: this step is skipped")

no weather in the us dataset: this step is skipped


### Step 19: project the stations, sort and filter (`denoise_weather`)

**What.** `x` and `y` (EPSG:32630) are added from the station positions, the table is sorted by (station, time) and
the numeric measurements are median-filtered per station with the same 30 s window as the AIS.

**Why.** The same filter keeps one definition of the cleaning step for every dataset; the stations' `x`, `y` are in
the CRS of the AIS positions. With hourly observations a 30 s window holds a single observation, so **the
weather values do not change**: the cell checks this.

**Source.** `catraj.preprocessing.filtering`.

In [26]:
if "weather" in spec.has:
    n0, before_w = len(rep), obs.sort_values(["station_id", "unix_time"], kind="stable").reset_index(drop=True)
    obs = steps.denoise_weather(obs, 30, rep)
    display(show(rep, n0))
    cols = list(steps.WEATHER_FILTERED)
    same = np.isclose(obs[cols].to_numpy(dtype=float), before_w[cols].to_numpy(dtype=float), equal_nan=True)
    print("measurements unchanged by the filter:", bool(same.all()), f"({same.size:,} values compared)")
    display(obs[["station_id", "unix_time", "air_temperature", "wind_speed", "x", "y"]].head(3))
    print(f"{len(obs)} rows = {obs.station_id.nunique()} stations x {len(obs) // obs.station_id.nunique()} hours")
else:
    print(f"no weather in the {DATASET} dataset: this step is skipped")

no weather in the us dataset: this step is skipped


## 1.3 Ocean (sea state; datasets with sea state: Zenodo)

`oc_<month>.csv` holds a WAVEWATCH III forecast (SHOM-IFREMER) on a grid of about 0.033 degrees, every 3 hours:
water depth, sea level, significant wave height, wave length and wave direction. The US dataset has none.

### Step 20: read the file with descriptive names (`read_ocean_raw`)

`dpt` becomes `bottom_depth`, `wlv` `sea_level`, `hs` `wave_height`, `lm` `wave_length`, `dir` `wave_direction`
and `ts` `unix_time`.

In [27]:
if "ocean" in spec.has:
    n0 = len(rep)
    oc = steps.read_ocean_raw(RAW / "E1_ocean" / "oc_november.csv", rep)
    display(show(rep, n0))
    display(oc.head(3))
    print(f"{oc[['lon', 'lat']].drop_duplicates().shape[0]:,} grid cells x {oc.unix_time.nunique()} time steps")
else:
    print(f"no ocean in the {DATASET} dataset: this step is skipped")

no ocean in the us dataset: this step is skipped


### Step 21: does the file belong to its month? (`ocean_month_matches`)

**What.** The month of a file is checked against its times (the median time must fall in the month of the file
name). A file whose times are not in its month is **skipped**, not written under the wrong name.

**Why.** The archive's `oc_january.csv` is a byte-for-byte copy of `oc_november.csv`: there is no sea state for
January 2016. Without this check the November data would be saved as January and used for January's ships. (Together
with the January exclusion of step 5, no January data enters the preprocessed archive.)

**Source.** The archive's files; the MD5 of the zip is the one Zenodo publishes. If the full archive is downloaded
the cell compares the two files byte by byte.

In [28]:
if "ocean" in spec.has:
    print("median time of the sample's file:", when(oc.unix_time.median()))
    for month in ("november", "january", "december"):
        print(f"  is it oc_{month}.csv?", steps.ocean_month_matches(oc, month))

    jan, nov = FULL_RAW / "E1_ocean" / "oc_january.csv", FULL_RAW / "E1_ocean" / "oc_november.csv"
    if jan.exists() and nov.exists():
        def md5(path):
            h = hashlib.md5()
            with open(path, "rb") as f:
                while chunk := f.read(16 << 20):
                    h.update(chunk)
            return h.hexdigest()
        print("\nfull archive:  md5 of oc_november.csv", md5(nov), "\n               md5 of oc_january.csv ", md5(jan))
    else:
        print("\n(the full archive is not downloaded here: Part 2; its oc_january.csv is identical to oc_november.csv)")
else:
    print(f"no ocean in the {DATASET} dataset: this step is skipped")

no ocean in the us dataset: this step is skipped


### Step 22: inside the area and the period; January out (`drop_invalid_ocean`, `drop_excluded_periods`)

Rows whose time is outside the archive's period, or whose cell is outside the archive's area, are dropped; the
January exclusion is applied to the sea-state rows as well. The sample was cut to the area, so nothing goes.

In [29]:
if "ocean" in spec.has:
    n0 = len(rep)
    oc = steps.drop_invalid_ocean(oc, spec.area, rep)
    oc = steps.drop_excluded_periods(oc, steps.EXCLUDED_PERIODS, rep, dataset="ocean")
    display(show(rep, n0))
else:
    print(f"no ocean in the {DATASET} dataset: this step is skipped")

no ocean in the us dataset: this step is skipped


### Step 23: land cells (`drop_land_cells`)

**What.** A cell whose `bottom_depth` is -16000 or less is land (the undefined depth is -16384): its rows are
dropped.

**Why.** The model has no sea state on land: **all the other fields of those rows are undefined** (the codes
below). Keeping them would mean thousands of rows of nothing. A cell never changes its depth, so a land cell is land at
every time step.

**Source.** `E1_ocean/README_parameters.txt` (undefined depth -16384).

The cell counts the undefined codes in the land rows and in the sea rows before dropping, and maps the cells.

In [30]:
if "ocean" in spec.has:
    land = oc.bottom_depth <= -16000
    codes = {"bottom_depth": -16384, "sea_level": -327.67, "wave_height": -65.534, "wave_length": -32767, "wave_direction": -3276.7}
    display(pd.DataFrame({c: {"in land rows": int(np.isclose(oc.loc[land, c], v).sum()),
                             "in sea rows": int(np.isclose(oc.loc[~land, c], v).sum())} for c, v in codes.items()}))
    print(f"{int(land.sum()):,} of {len(oc):,} rows are land ({land.mean():.1%})")

    first = oc[oc.unix_time == oc.unix_time.min()]
    sea = first.bottom_depth > -16000
    print(f"{len(first):,} grid cells in this time step: {int(sea.sum()):,} sea, {int((~sea).sum()):,} land")
    show_n = 10000                                   # at most 10,000 cells of each kind are drawn
    sea_cells = first[sea].sample(min(int(sea.sum()), show_n), random_state=0).assign(depth=lambda d: d.bottom_depth)
    land_cells = first[~sea].sample(min(int((~sea).sum()), show_n), random_state=0)
    land_map = land_cells.hvplot.points(x="lon", y="lat", geo=True, color="#999999", size=3, label="land cell (dropped)", alpha=.7)
    sea_map = sea_cells.hvplot.points(x="lon", y="lat", geo=True, c="depth", cmap="Blues_r", size=3, clabel="depth (m)",
                                      hover_cols=["bottom_depth"], colorbar=True)
    ocean_fig = (gv.tile_sources.CartoLight * land_map * sea_map).opts(
        title="one time step of the sample's sea state", width=620, height=520, legend_position="bottom_left",
        active_tools=["wheel_zoom"])
    display(ocean_fig)

    n0 = len(rep)
    oc = steps.drop_land_cells(oc, rep)
    display(show(rep, n0))
else:
    print(f"no ocean in the {DATASET} dataset: this step is skipped")

no ocean in the us dataset: this step is skipped


### Step 24: undefined values become NaN (`ocean_unavailable_to_nan`)

**What.** From the README, the codes for "undefined" are: depth -16384 (handled by step 23), sea level
-327.67, wave height -65.534, wave length -32767, wave direction -3276.7. A negative wave height, length or direction,
and a sea level of -327 or less, become NaN.

**Why.** A negative height or length is not physical. The sea level may be negative (a tide), so only its code,
-327.67, is undefined: that is why the rule is "-327 or less" and not "below zero".

**Source.** `E1_ocean/README_parameters.txt`.

On the sea cells that remain, only a few wave lengths and directions are undefined (the counts below): all the other
codes belonged to land rows.

In [31]:
if "ocean" in spec.has:
    n0 = len(rep)
    oc = steps.ocean_unavailable_to_nan(oc, rep)
    display(show(rep, n0))
    print(oc[["sea_level", "wave_height", "wave_length", "wave_direction"]].isna().sum().to_string())
else:
    print(f"no ocean in the {DATASET} dataset: this step is skipped")

no ocean in the us dataset: this step is skipped


### Step 25: physical ranges (`check_ocean_ranges`)

**What.** A value outside its range becomes NaN: depth at most 11000 m (the deepest ocean), sea level -20 to 20 m,
wave height 0 to 30 m, wave length 0 to 2000 m, wave direction 0 <= v < 360 degrees.

**Why.** To catch errors without discarding real extremes (the largest waves measured are about 30 m).
**Source.** World records, in the README's units.

In [32]:
if "ocean" in spec.has:
    n0 = len(rep)
    oc = steps.check_ocean_ranges(oc, rep)
    display(show(rep, n0))
    display(oc[["bottom_depth", "sea_level", "wave_height", "wave_length", "wave_direction"]].describe().loc[["count", "min", "max"]].round(2))
else:
    print(f"no ocean in the {DATASET} dataset: this step is skipped")

no ocean in the us dataset: this step is skipped


### Step 26: sort by grid point and time (`sort_ocean`), step 27: median filter (`denoise_ocean`)

`sort_ocean` sorts by `(lon, lat, unix_time)` so that each grid point's series is contiguous.
`denoise_ocean` then filters each grid point's series (sea level, wave height, length, and direction on the circle) with
the same 30 s median. The forecast is three-hourly, so a window holds one value and **the series do not change**; the
step exists so every dataset goes through the same definition (and so that a different `window` works for all).

In [33]:
if "ocean" in spec.has:
    n0 = len(rep)
    oc = steps.sort_ocean(oc, rep)
    before_o = oc.copy()
    oc = steps.denoise_ocean(oc, 30, rep)
    display(show(rep, n0))
    cols = ["sea_level", "wave_height", "wave_length", "wave_direction"]
    print("values unchanged by the filter:", bool(np.isclose(oc[cols].to_numpy(dtype=float), before_o[cols].to_numpy(dtype=float), equal_nan=True).all()))
    display(oc.head(3))
else:
    print(f"no ocean in the {DATASET} dataset: this step is skipped")

no ocean in the us dataset: this step is skipped


## 1.4 Coastline

*Zenodo*: `C2_European_Coastline` is the EEA's polygons of European land ("EEA coastline for analysis"), in EPSG:3035. *US*: the NOAA Medium
Resolution Shoreline, `us_medium_shoreline.shp`, **lines** in EPSG:4269 (NAD83), of which the regions N (New England), M (Mid-Atlantic), S (South
Atlantic), GS and SG (Florida Keys) and MN and NM (border) are kept: the lines are the barriers of the topology-preserving compression.

### Steps 28 to 30: read, project, clip (`read_coastline`, `project_coastline`, `clip_coastline`)

**What.** (28) The polygons or lines that touch the dataset's area are read (the US area is widened by 0.5 degrees, so that a line
just outside is still a barrier). (29) They are reprojected from the file's CRS to the dataset's CRS, **the CRS of the AIS positions**. (30) They are
clipped to the area's outline (Zenodo: a rectangle in longitude and latitude, densified so its edges curve in UTM; US: the study-area polygon plus 0.5
degrees, projected), and empty results are dropped.

**Why.** The positions and the land must be in one CRS to ask "is this point on land?" (notebook 03/04, the
topology-preserving compression). The coastline of all Europe, or of all the US, is large; only the area is kept.

**Source.** EEA for Zenodo (acknowledge the source: Copyright holder DG ENV, http://www.eea.europa.eu/legal/copyright); NOAA, Medium Resolution
Shoreline (<https://shoreline.noaa.gov/data/datasheets/medium.html>).

In [34]:
if "coastline" in spec.has:
    shp = spec.coastline_path(RAW)
    print("raw file:", rel(shp))
    n0 = len(rep)
    coast = steps.read_coastline(shp, spec.area, rep)
    print("as read: CRS", coast.crs.to_string(), "| geometries:", len(coast), f"({coast.geom_type.iloc[0]})")
    coast = steps.project_coastline(coast, CRS, rep)
    print("projected: CRS", coast.crs.to_string())
    coast = steps.clip_coastline(coast, spec.area, rep)
    display(show(rep, n0))

    pick = ais.sample(min(len(ais), 20000), random_state=0)
    if coast.geom_type.iloc[0].startswith("Polygon"):
        coast_map = coast.hvplot.polygons(geo=True, crs=UTM, color="#d9d3c4", line_color="#777777", line_width=.5,
                                          hover=False, label="coastline (clipped)")
    else:
        coast_map = coast.hvplot.paths(geo=True, crs=UTM, color="#444444", line_width=1.2, hover=False,
                                       label="shoreline (clipped)")
    ais_map = pick.hvplot.points(x="x", y="y", geo=True, crs=UTM, color="#1f77b4", size=2, alpha=.6,
                                 label=f"AIS positions ({len(pick):,} of them)", hover=False)
    display((gv.tile_sources.CartoLight * coast_map * ais_map).opts(
        title="coastline (clipped) and the sample's positions", width=700, height=560, legend_position="bottom_left",
        active_tools=["wheel_zoom"]))
else:
    print("no coastline in this dataset")

raw file: data/UsAIS/sample/raw/shoreline/us_medium_shoreline.shp
as read: CRS EPSG:4269 | geometries: 550 (LineString)
projected: CRS EPSG:32618


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,read_coastline,lines touching the area,550,550,0,0
1,read_coastline,"regions N,M,S,GS,SG,MN,NM",550,550,0,0
2,project_coastline,to EPSG:32618,550,550,0,0
3,clip_coastline,clipped to the area + 0.5 degrees,550,550,0,0


:Overlay
   .WMTS.I                                                                       :WMTS   [Longitude,Latitude]
   .Path.Shoreline_left_parenthesis_clipped_right_parenthesis                    :Path   [Longitude,Latitude]
   .Points.AIS_positions_left_parenthesis_20_comma_000_of_them_right_parenthesis :Points   [x,y]

### The study area, the sample's region and the shoreline

The same map for both datasets, in longitude and latitude: the **area** the dataset is cut to (Zenodo: the archive's box; US: the east-coast polygon, the
box minus the Gulf side of Florida and minus the Great Lakes), the **region of the sample** (the bounding box of the sample's positions), and the **shoreline or
coastline**. If the full preprocessed shoreline exists on this machine (`data/.../preprocessed/Coastline/`, after Part 3) it is drawn, simplified, so that you see
the whole coast; otherwise the sample's own clipped lines are drawn.

In [35]:
area_poly = spec.area.polygon if hasattr(spec.area, "polygon") else box(spec.area[0], spec.area[2], spec.area[1], spec.area[3])
area_gdf = gpd.GeoDataFrame({"layer": ["study area"]}, geometry=[area_poly], crs=4326)
region_gdf = gpd.GeoDataFrame({"layer": ["sample region"]}, geometry=[box(ais.lon.min(), ais.lat.min(), ais.lon.max(), ais.lat.max())], crs=4326)

full_files = sorted((FULL_OUT / "Coastline").glob("*.parquet"))
if full_files:
    shore = gpd.read_parquet(full_files[0])
    which = f"full shoreline ({rel(full_files[0])}), simplified"
else:
    shore = coast.copy()
    which = "the sample's shoreline"
shore = shore.to_crs(4326)
shore["geometry"] = shore.geometry.simplify(0.02)
shore = shore[~shore.geometry.is_empty]
if not shore.geom_type.iloc[0].startswith("Polygon"):
    shore = shore[shore.to_crs(CRS).length > 15_000]    # drop the tiny islands and the short pieces (under 15 km): a map, not a barrier
print(f"{which}: {len(shore):,} geometries drawn")

if shore.geom_type.iloc[0].startswith("Polygon"):
    shore_layer = shore.hvplot.polygons(geo=True, crs=PLATE, color="#d9d3c4", line_color="#777777", line_width=.5, hover=False, label="coastline")
else:
    shore_layer = shore.hvplot.paths(geo=True, crs=PLATE, color="#444444", line_width=1, hover=False, label="shoreline")
area_layer = area_gdf.hvplot.polygons(geo=True, crs=PLATE, color="#d62728", fill_alpha=.06, line_color="#d62728", line_width=2, hover=False, label="study area")
region_layer = region_gdf.hvplot.polygons(geo=True, crs=PLATE, color="#1f77b4", fill_alpha=.25, line_color="#1f77b4", line_width=2, hover=False, label="sample region")
display((gv.tile_sources.CartoLight * shore_layer * area_layer * region_layer).opts(
    title=f"{spec.title}: study area, sample region and shoreline", width=760, height=640,
    legend_position="bottom_left", active_tools=["wheel_zoom"]))

full shoreline (data/UsAIS/preprocessed/Coastline/shoreline.parquet), simplified: 496 geometries drawn


:Overlay
   .WMTS.I                 :WMTS   [Longitude,Latitude]
   .Path.Shoreline         :Path   [Longitude,Latitude]
   .Polygons.Study_area    :Polygons   [Longitude,Latitude]
   .Polygons.Sample_region :Polygons   [Longitude,Latitude]

## 1.5 Write the result: `spec.preprocess`

`spec.preprocess` runs all of the steps above (with the same functions, in the same order) on everything the dataset has and writes the
files other notebooks read through `catraj.datasets`:

| | Zenodo (`preprocess_archive`) | US (`preprocess_us`) |
|---|---|---|
| AIS | `AIS/ais.parquet`, `AIS/mmsi_check.csv` | `AIS/ais_<month>.parquet`, one file per month (`spec.ais_outputs(out)`) |
| Weather | `Weather/weather.parquet` | none |
| Sea state | `Ocean/ocean_november.parquet` | none |
| Coastline | `Coastline/coastline.parquet` (polygons, GeoParquet, EPSG:32630) | `Coastline/shoreline.parquet` (lines, GeoParquet, EPSG:32618) |
| Report | `preprocessing_report.csv`, `.txt`: rows in and out and values set to NaN of every rule | the same |

It writes to the dataset's `sample/preprocessed/`, the **committed** folder, with `force=True`. The result is deterministic, so running it
again reproduces the committed files: the cell hashes every file before and after and says whether anything changed (`git status` of the
folder shows the same).

In [36]:
def digests(folder):
    return {p.relative_to(folder).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(folder.rglob("*")) if p.is_file()}

committed = digests(OUT)
t = time.perf_counter()
report = spec.preprocess(raw_root=RAW, out_root=OUT, force=True, n_jobs=N_JOBS)
print(f"\nwritten in {time.perf_counter() - t:.1f} s")
after = digests(OUT)
changed = [f for f in after if committed.get(f) != after[f]] + [f for f in committed if f not in after]
print("files that differ from the committed ones:", changed if changed else "none (reproduced byte for byte)")

2017-11: 2 daily files ...


  AIS 2017-11: 2 days read in 8 s


  AIS 2017-11: 1/1 batches filtered
  2017-11: 465,795 rows, 507 vessels -> C:\dev\CATraj\data\UsAIS\sample\preprocessed\AIS\ais_2017-11.parquet (10 s)
skip 2017-12: no ais-2017-12-DD.csv.zst in C:\dev\CATraj\data\UsAIS\sample\raw


  shoreline: 550 lines -> C:\dev\CATraj\data\UsAIS\sample\preprocessed\Coastline\shoreline.parquet

written in 11.1 s
files that differ from the committed ones: none (reproduced byte for byte)


### The report

The report is the record of every rule: rows in, rows out, rows dropped, values set to NaN. It is written beside the
data as `preprocessing_report.csv` and `.txt`. The text below is what `spec.preprocess` wrote; the check after it compares
the written report with what the cells above recorded one step at a time (line by line, in any order: `preprocess_us` reports per month, as
`AIS 2017-11`, and runs the vessel types before the position jumps, see step 10; so for `attach_vessel_types` only the NaN count is compared, not the rows in and out; and the US workers do not run the study-area test of step 6, which the reader has done).

In [37]:
print((OUT / "preprocessing_report.txt").read_text())
cols = ["step", "rows_in", "rows_out", "values_to_nan"]      # the rule texts of the US month workers differ a little (e.g. the filter line)
def lines(frame):
    frame = frame[~frame["rule"].str.contains("inside study area")].copy()     # US: the reader clipped already, the month workers skip this rule
    frame["dataset"] = frame["dataset"].str.split().str[0]        # "AIS 2017-11" -> "AIS"
    typed = frame["step"] == "attach_vessel_types"                # US: run before the jumps, so rows in/out differ by the jump rows
    frame.loc[typed, ["rows_in", "rows_out"]] = 0
    return sorted(map(tuple, frame[["dataset"] + cols].astype(str).to_numpy()))
mine = rep.to_frame().reset_index(drop=True)
written = pd.read_csv(OUT / "preprocessing_report.csv")
same_report = lines(mine) == lines(written)
print("\nthe step-by-step cells recorded the written report's lines (rows in/out and NaN counts):", bool(same_report))
if not same_report:
    only_mine = [l for l in lines(mine) if l not in lines(written)]
    only_written = [l for l in lines(written) if l not in lines(mine)]
    print("only in the cells' record:", only_mine, "\nonly in the written report:", only_written)

    dataset                     step                                                                                      rule  rows_in  rows_out  rows_dropped  values_to_nan
AIS 2017-11             read_ais_raw                                                                                 rows read   465852    465852             0              0
AIS 2017-11             read_ais_raw                                                            position inside the study area   465852    465852             0              0
AIS 2017-11             read_ais_raw information only, kept: of these, lat > 44.5 and lon < -73 (St. Lawrence, Lake Champlain)        0         0             0              0
AIS 2017-11        drop_invalid_mmsi                                                                   mmsi: ship station kept   465852    465798            54              0
AIS 2017-11        drop_invalid_mmsi                                                              mmsi: craft of a parent shi

In [38]:
# the step-by-step tables equal the written files
loaded = pd.concat([pd.read_parquet(f) for f in spec.ais_outputs(OUT)], ignore_index=True)
cols = list(ais.columns)
pd.testing.assert_frame_equal(loaded[cols], ais[cols].reset_index(drop=True), check_dtype=False, check_categorical=False)
print("AIS: the cells' table equals", ", ".join(f.name for f in spec.ais_outputs(OUT)), ":", len(loaded), "rows,", loaded.mmsi.nunique(), "vessels")
if "weather" in spec.has:
    w = pd.read_parquet(OUT / "Weather" / "weather.parquet")
    pd.testing.assert_frame_equal(w[list(obs.columns)], obs.reset_index(drop=True), check_dtype=False)
    print("weather: equals weather.parquet:", len(w), "rows")
if "ocean" in spec.has:
    o = pd.read_parquet(OUT / "Ocean" / "ocean_november.parquet")
    pd.testing.assert_frame_equal(o[list(oc.columns)], oc.reset_index(drop=True), check_dtype=False)
    print("ocean: equals ocean_november.parquet:", len(o), "rows")
c = gpd.read_parquet(next((OUT / "Coastline").glob("*.parquet")))
print("coastline:", len(c), "geometries, CRS", c.crs.to_epsg(), "| the cells' table:", len(coast), "geometries")

print()
for f, size in [(p.relative_to(OUT).as_posix(), p.stat().st_size) for p in sorted(OUT.rglob("*")) if p.is_file()]:
    print(f"{size / 1e6:8.2f} MB  {f}")

AIS: the cells' table equals ais_2017-11.parquet : 465795 rows, 507 vessels
coastline: 550 geometries, CRS 32618 | the cells' table: 550 geometries

   11.69 MB  AIS/ais_2017-11.parquet
    0.47 MB  Coastline/shoreline.parquet
    0.00 MB  preprocessing_report.csv
    0.00 MB  preprocessing_report.txt


# Part 2 - Download the full data, file by file

The samples are two-day cuts. The full data is public.

* **Zenodo**: <https://zenodo.org/records/1167595> (Ray et al., 2019, *Heterogeneous integrated dataset for Maritime Intelligence, Surveillance, and
  Reconnaissance*, Data in Brief 25, 104141; data doi:10.5281/zenodo.1167595). Five zip files, 825 MB in all, extracted into `data/ZenodoData/`: about 5.3 GB of
  CSV, 4 GB of it the six months of sea state (each `oc_<month>.csv` is about 0.67 GB).
* **US**: the 61 daily files `ais-2017-11-01.csv.zst` ... `ais-2017-12-31.csv.zst` from
  <https://noaaocm.blob.core.windows.net/ais/csv2/csv2017/> (index page: <https://noaaocm.blob.core.windows.net/ais/csv2/csv2017/index.html>; catalogue:
  <https://hub.marinecadastre.gov>), 100 to 250 MB each, about 12.5 GB in all, into `data/UsAIS/csv2017/`; and the NOAA Medium Resolution Shoreline,
  <https://coast.noaa.gov/htdata/Shoreline/us_medium_shoreline.zip> (14 MB), into `data/UsAIS/shoreline/`.

**How long it takes** depends on your connection: for Zenodo at 10 MB/s about 1.5 minutes (preprocessing, Part 3, takes about 2 minutes more); for the US data
at 10 MB/s about 20 minutes.

**How.** `DOWNLOAD = True` (settings cell) makes the cell call the dataset's script, `scripts/download_data.py` (Zenodo) or `scripts/download_us.py` (US): the notebook
imports it and uses its functions instead of repeating its logic. For each file, one at a time:

* it streams the file with a progress line (MB done / total, MB/s). An interrupted download **resumes** where it stopped (a `.part` file and an HTTP range
  request) and is retried up to five times;
* it checks the file against the checksum the source publishes (the MD5 on the Zenodo record; the MD5 and size the NOAA server reports) and deletes a corrupted
  download, which must then be repeated;
* Zenodo: it extracts the files the code reads into `data/ZenodoData/<folder>/`; US: the daily files are used as they are, and the shoreline zip is extracted.

**Never overwritten.** A file already in place is skipped ("already in place"): nothing is deleted or downloaded again (the Zenodo zips are kept,
`remove_zip=False`; the script's `--remove-zips` would delete them).

**Licences** (details in `data/LICENSE.md`). *Zenodo*: the AIS data is **CC BY-NC-SA 4.0** (attribute NARI, Brest; no commercial use; derived data under the same
licence); the sea state (SHOM-IFREMER, WAVEWATCH III) is public with the citation mandatory (Boudiere et al., 2013, doi:10.1016/j.ijome.2013.11.010); the weather
(rp5.ua, from Met Office / NOAA data) is public, cite the source; the European coastline (EEA) may be re-used free of charge when the source is acknowledged.
*US*: MarineCadastre.gov / NOAA data are public (a US Government work); cite "MarineCadastre.gov, AIS data for 2017" and the NOAA shoreline. The code's BSD
licence does not cover the data.

The same can be done in a terminal: `python scripts/download_data.py` (Zenodo) or `python scripts/download_us.py` (US), each with `--check` for the report.

In [39]:
def load_script(name):
    "import scripts/<name>.py as a module (its functions; nothing is run)"
    s = importlib.util.spec_from_file_location(name, ROOT / "scripts" / f"{name}.py")
    module = importlib.util.module_from_spec(s)
    s.loader.exec_module(module)
    return module

if DATASET == "zenodo":
    dl = load_script("download_data")
    MONTHS = list(dl.MONTHS)                    # oct, nov, dec, jan, feb, mar: every sea-state month (Part 3 skips January)
    NAMES = dl.TARGETS["all"]
    table = pd.DataFrame([{"zenodo file": n, "size (MB)": dl.ARCHIVES[n]["size"] / 1e6, "extracted to": dl.ARCHIVES[n]["folder"] + "/",
                           "in place": dl.present(n, MONTHS)} for n in NAMES])
    display(table.round(3))
    print(f"total download: {table['size (MB)'].sum():,.0f} MB; missing now: "
          f"{table.loc[~table['in place'], 'size (MB)'].sum():,.0f} MB")
else:
    dl = load_script("download_us")
    days = pd.Series(dl.present_days(), name="in place")
    per_month = days.groupby(days.index.str[4:11]).agg(days_in_place="sum", days="size")
    per_month["size (GB)"] = [sum((dl.AIS_DIR / n).stat().st_size for n in days.index[days.index.str[4:11] == m] if days[n]) / 1e9
                              for m in per_month.index]
    display(per_month.round(2))
    print(f"{len(days)} daily files ({days.sum()} in place, {int((~days).sum())} missing); shoreline in place: {dl.shoreline_present()}")
    print("from", dl.AIS_URL, "and", dl.SHORELINE_URL)

,days_in_place,days,size (GB)
2017-11,30,30,6.41
2017-12,31,31,6.12


61 daily files (61 in place, 0 missing); shoreline in place: True
from https://noaaocm.blob.core.windows.net/ais/csv2/csv2017/ and https://coast.noaa.gov/htdata/Shoreline/us_medium_shoreline.zip


In [40]:
if DOWNLOAD:
    t = time.perf_counter()
    if DATASET == "zenodo":
        dl.DATA.mkdir(parents=True, exist_ok=True)
        for name in NAMES:                      # one file at a time; download_data.fetch prints its progress
            t1 = time.perf_counter()
            dl.fetch(name, MONTHS, remove_zip=False)
            print(f"  {time.perf_counter() - t1:.1f} s")
    else:
        dl.fetch_days()                         # the 61 days, one by one; files in place are skipped
        dl.fetch_shoreline()                    # the NOAA shoreline; skipped if a shapefile is in place
    print(f"\ndownload step finished in {time.perf_counter() - t:.0f} s")
else:
    print("DOWNLOAD is False: nothing is downloaded. Set DOWNLOAD = True in the settings cell to fetch what is missing.")

DOWNLOAD is False: nothing is downloaded. Set DOWNLOAD = True in the settings cell to fetch what is missing.


### What is present (the `--check` report)

`report` is what `python scripts/download_data.py --check` (or `download_us.py --check`) prints: for Zenodo one line per file, marked when everything the code reads from it is
extracted, and the sea-state months; for the US data the number of daily files and the shoreline.

In [41]:
if DATASET == "zenodo":
    print(f"data for 'all' in {dl.DATA.relative_to(dl.ROOT).as_posix()}/ (every sea-state month):")
    complete = dl.report(NAMES, MONTHS)
    print("\ncomplete:", complete)
    print("\nsea-state files:")
    for m, full_name in dl.MONTHS.items():
        f = dl.DATA / "E1_ocean" / f"oc_{full_name}.csv"
        print(f"  [{'x' if f.exists() else ' '}] oc_{full_name}.csv" + (f"   {f.stat().st_size / 1e6:,.0f} MB" if f.exists() else ""))
else:
    print("data for 2017-11 and 2017-12 in data/UsAIS/:")
    complete = dl.report()
    print("\ncomplete:", complete)

data for 2017-11 and 2017-12 in data/UsAIS/:
  AIS days: 61/61 in C:\dev\CATraj\data\UsAIS\csv2017  (12.5 GB)
  shoreline: present in C:\dev\CATraj\data\UsAIS\shoreline

complete: True


# Part 3 - Preprocess the full data

`spec.preprocess()` (no arguments: the full raw data to the dataset's `preprocessed/` folder) runs the steps of Part 1 in the same order (it is the very code the
cells above called one by one) on everything the dataset has.

* *Zenodo* (`preprocess_archive(data/ZenodoData, data/ZenodoData/preprocessed)`): on every dataset it finds. It writes `AIS/ais.parquet`, `Weather/weather.parquet`, one
  `Ocean/ocean_<month>.parquet` per month and `Coastline/coastline.parquet`, plus the report. Months are skipped when their raw file is absent, and **January 2016 is
  excluded** (its AIS and weather rows are dropped and its `oc_january.csv`, a copy of November, is skipped). About **2 minutes** for the whole archive (19 million AIS
  messages): 97 to 151 s serially and 82 to 89 s on 16 cores. Terminal: `python scripts/prepare_zenodo.py --n-jobs -1`.
* *US* (`preprocess_us(data/UsAIS/csv2017, data/UsAIS/preprocessed)`): month by month. It writes `AIS/ais_2017-11.parquet` and `AIS/ais_2017-12.parquet` (about 60 and 55 million positions
  of the 61 daily files, 466 million messages read), `Coastline/shoreline.parquet` and the report, with the report's `dataset` column naming the month (`AIS 2017-11`). It takes
  much longer than Zenodo's (a month is about 4 to 5 GB as a frame; see `scripts/prepare_us.py`). Terminal: `python scripts/prepare_us.py --n-jobs -1`.

A dataset whose output exists is skipped unless `FORCE_FULL = True`. **When the flag is off** (the committed state), the cell loads the report of an existing full run and shows it, so the
numbers below are those of the full data. If no full output exists here, it says so. The raw files are only read; the output is not committed.

In [42]:
raw_here = any(FULL_RAW.glob("ais-*.csv*")) if DATASET == "us" else spec.ais_path(FULL_RAW).exists()
if PREPROCESS_FULL:
    if not raw_here:
        print("the full raw data is not downloaded: set DOWNLOAD = True (Part 2) first")
    else:
        t = time.perf_counter()
        spec.preprocess(raw_root=FULL_RAW, out_root=FULL_OUT, force=FORCE_FULL, n_jobs=N_JOBS)
        print(f"preprocessed in {time.perf_counter() - t:.0f} s (n_jobs={N_JOBS})")
else:
    print("PREPROCESS_FULL is False: nothing is run here.")

full_report_path = FULL_OUT / "preprocessing_report.csv"
if full_report_path.exists():
    full = pd.read_csv(full_report_path)
    full["family"] = full["dataset"].str.split().str[0]
    stamp = pd.Timestamp(full_report_path.stat().st_mtime, unit="s", tz="UTC").strftime("%Y-%m-%d %H:%M UTC")
    print(f"\nloaded the report of the full {DATASET} data ({stamp}): {len(full)} lines, datasets: {', '.join(full.dataset.unique())}")
else:
    full = None
    print("\nno full output here (", FULL_OUT, "): run Parts 2 and 3 with the flags on.")

PREPROCESS_FULL is False: nothing is run here.

loaded the report of the full us data (2026-11-25 17:53 UTC): 64 lines, datasets: AIS 2017-11, AIS 2017-12, coastline


### The full report

One line per rule (the dataset, the step, the rule, the rows in and out, the rows dropped and the values set to NaN): the same record as for the
sample, now on the full data. It is a long table, so the cells show the AIS lines (for the US data, one block per month) and the other datasets' lines apart.

In [43]:
if full is not None:
    cols = ["step", "rule", "rows_in", "rows_out", "rows_dropped", "values_to_nan"]
    pd.set_option("display.max_rows", 200)
    for label in full[full.family == "AIS"].dataset.unique():
        print(label)
        display(full[full.dataset == label][cols].reset_index(drop=True))

AIS 2017-11


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,read_ais_raw,rows read,235903781,235903781,0,0
1,read_ais_raw,position inside the study area,235903781,60047675,175856106,0
2,read_ais_raw,"information only, kept: of these, lat > 44.5 and lon < -73 (St. Lawrence, La...",736538,736538,0,0
3,drop_invalid_mmsi,"mmsi: ship, unallocated MID",139445,0,139445,0
4,drop_invalid_mmsi,mmsi: placeholder,37142,0,37142,0
5,drop_invalid_mmsi,mmsi: handheld,38022,0,38022,0
6,drop_invalid_mmsi,mmsi: craft of a parent ship,5392,0,5392,0
7,drop_invalid_mmsi,mmsi: unassigned pattern,60376,0,60376,0
8,drop_invalid_mmsi,"mmsi: craft of a parent ship, unallocated MID",199,0,199,0
9,drop_invalid_mmsi,mmsi: distress device (SART/MOB/EPIRB),172,0,172,0


AIS 2017-12


,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,read_ais_raw,rows read,229994263,229994263,0,0
1,read_ais_raw,position inside the study area,229994263,55508638,174485625,0
2,read_ais_raw,"information only, kept: of these, lat > 44.5 and lon < -73 (St. Lawrence, La...",678770,678770,0,0
3,drop_invalid_mmsi,mmsi: unassigned pattern,41196,0,41196,0
4,drop_invalid_mmsi,"mmsi: ship, unallocated MID",107742,0,107742,0
5,drop_invalid_mmsi,mmsi: handheld,40385,0,40385,0
6,drop_invalid_mmsi,mmsi: placeholder,38531,0,38531,0
7,drop_invalid_mmsi,"mmsi: aid to navigation, unallocated MID",26,0,26,0
8,drop_invalid_mmsi,mmsi: craft of a parent ship,4039,0,4039,0
9,drop_invalid_mmsi,"mmsi: craft of a parent ship, unallocated MID",97,0,97,0


In [44]:
if full is not None and (full.family == "weather").any():
    print("weather")
    display(full[full.family == "weather"][cols].reset_index(drop=True))
elif full is not None:
    print(f"no weather in the {DATASET} dataset")

no weather in the us dataset


In [45]:
if full is not None:
    print("ocean and coastline" if "ocean" in spec.has else "coastline")
    display(full[full.family.isin(["ocean", "coastline"])][["dataset"] + cols].reset_index(drop=True))

coastline


,dataset,step,rule,rows_in,rows_out,rows_dropped,values_to_nan
0,coastline,read_coastline,lines touching the area,30378,30378,0,0
1,coastline,read_coastline,"regions N,M,S,GS,SG,MN,NM",30378,25061,5317,0
2,coastline,project_coastline,to EPSG:32618,25061,25061,0,0
3,coastline,clip_coastline,clipped to the area + 0.5 degrees,24055,24073,-18,0


### Rows removed per rule

The AIS rows each rule removed from the full data (summed over the months), on a logarithmic axis because the rules differ by orders of magnitude. For the
US data the largest is the clip to the study area, which discards the positions of all other US waters at the moment they are read.

In [46]:
if full is not None:
    a = full[(full.family == "AIS") & (full.rows_dropped > 0) & ~full.rule.str.endswith("kept")]   # "ship station kept" repeats the classes' total
    drops = a.groupby(["step", "rule"], sort=False).rows_dropped.sum().reset_index()
    drops["rule"] = (drops.step + ": " + drops.rule).str.slice(0, 70)
    drops = drops.sort_values("rows_dropped")
    display(drops.rename(columns={"rows_dropped": "rows removed"})[["rule", "rows removed"]].iloc[::-1].reset_index(drop=True))
    display(drops.hvplot.barh(x="rule", y="rows_dropped", logx=True, color="#d62728", width=900, height=60 + 26 * len(drops),
                              xlabel="", ylabel="rows removed (log scale)", ylim=(1, None), title=f"{spec.title}: AIS rows removed per rule",
                              hover_cols=["rows_dropped"]))

,rule,rows removed
0,read_ais_raw: position inside the study area,350341731
1,"drop_invalid_mmsi: mmsi: ship, unallocated MID",247187
2,drop_invalid_mmsi: mmsi: unassigned pattern,101572
3,drop_invalid_mmsi: mmsi: handheld,78407
4,drop_invalid_mmsi: mmsi: placeholder,75673
5,drop_invalid_mmsi: mmsi: craft of a parent ship,9431
6,drop_position_jumps: isolated jump above 50 knots,9041
7,"drop_duplicates_and_sort: duplicate (mmsi, unix_time)",1690
8,"drop_invalid_mmsi: mmsi: craft of a parent ship, unallocated MID",296
9,"drop_invalid_mmsi: mmsi: SAR aircraft, unallocated MID",261


:Bars   [rule]   (rows_dropped)

### Summary per month, vessels and vessel types

The AIS rows and vessels per month, and per vessel type, are counted from the written Parquet files (only the `mmsi`, `unix_time` and `vessel_type` columns, so it is
fast). *Zenodo*: weather and sea-state rows per month too, and **January 2016 has no rows: it is excluded.** *US*: one file per month; a vessel's type is chosen per month, so a vessel
that changed type between the months is counted under both.

In [47]:
def scan_ais(files):
    "rows and vessels per month, and the vessel types, from the written AIS Parquet files"
    rows, vessel_sets, pairs, type_rows = {}, {}, [], {}
    for f in files:
        t = pd.read_parquet(f, columns=["mmsi", "unix_time", "vessel_type"])
        day = (t["unix_time"].to_numpy() // 86400)
        days_u, inverse = np.unique(day, return_inverse=True)
        months_u = pd.to_datetime(days_u * 86400, unit="s").strftime("%Y-%m")
        codes, names = pd.factorize(months_u)
        month = codes[inverse]
        for k, name in enumerate(names):
            sel = month == k
            rows[name] = rows.get(name, 0) + int(sel.sum())
            vessel_sets.setdefault(name, []).append(np.unique(t["mmsi"].to_numpy()[sel]))
        pairs.append(t[["mmsi", "vessel_type"]].drop_duplicates())
        for k, v in t.groupby("vessel_type", observed=True).size().items():
            type_rows[k] = type_rows.get(k, 0) + int(v)
        del t
    vessels = {m: len(np.unique(np.concatenate(v))) for m, v in vessel_sets.items()}
    all_vessels = len(np.unique(np.concatenate([np.concatenate(v) for v in vessel_sets.values()])))
    pairs = pd.concat(pairs).drop_duplicates()
    return rows, vessels, all_vessels, pairs, pd.Series(type_rows)

files = spec.ais_outputs(FULL_OUT) if full is not None else []
files = [f for f in files if f.exists()]
if files:
    t = time.perf_counter()
    rows_m, vessels_m, n_vessels, pairs, type_rows = scan_ais(files)
    summary = pd.DataFrame({"AIS rows": pd.Series(rows_m), "vessels": pd.Series(vessels_m)})
    total = {"AIS rows": sum(rows_m.values()), "vessels": n_vessels}
    if "weather" in spec.has and (FULL_OUT / "Weather" / "weather.parquet").exists():
        def per_month(df):
            return df.groupby(pd.to_datetime(df["unix_time"], unit="s", utc=True).dt.strftime("%Y-%m")).size()
        wx = pd.read_parquet(FULL_OUT / "Weather" / "weather.parquet", columns=["station_id", "unix_time"])
        summary["weather rows"] = per_month(wx)
        oc_rows = {}
        for f in sorted((FULL_OUT / "Ocean").glob("ocean_*.parquet")):
            for k, v in per_month(pd.read_parquet(f, columns=["unix_time"])).items():
                oc_rows[k] = oc_rows.get(k, 0) + int(v)
        summary["sea-state rows"] = pd.Series(oc_rows)
        total.update({"weather rows": len(wx), "sea-state rows": sum(oc_rows.values())})
    if DATASET == "zenodo":
        summary = summary.reindex(["2015-10", "2015-11", "2015-12", "2016-01", "2016-02", "2016-03"])
    summary = summary.sort_index().fillna(0).astype(int)
    summary.index.name = "month"
    display(pd.concat([summary, pd.DataFrame(total, index=["all months"])]).map("{:,}".format))
    print(f"scanned in {time.perf_counter() - t:.0f} s")
else:
    print("no full output to summarise")

,AIS rows,vessels
2017-11,"59,759,561","10,974"
2017-12,"55,272,838","9,301"
all months,"115,032,399","13,261"


scanned in 10 s


In [48]:
if files:
    by_type = pd.DataFrame({"vessels": pairs.groupby("vessel_type", observed=True).mmsi.nunique(), "positions": type_rows}).fillna(0).astype(int)
    by_type = by_type.sort_values("positions", ascending=False)
    display(by_type.map("{:,}".format))
    top = by_type.head(15).reset_index().rename(columns={"index": "vessel_type"})
    v = top.sort_values("vessels").hvplot.barh(x="vessel_type", y="vessels", color="#1f77b4", title="vessels per type", width=450, height=420, xlabel="", hover_cols=["positions"])
    p = top.sort_values("positions").hvplot.barh(x="vessel_type", y="positions", color="#ff7f0e", logx=True, ylim=(1, None), title="positions per type (log scale)", width=450, height=420, xlabel="", hover_cols=["vessels"])
    display((v + p).cols(2))

,vessels,positions
Unknown,"3,899","45,553,647"
Pleasure Craft,"4,379","27,294,609"
Cargo,"1,892","10,253,508"
Passenger,619,"10,032,007"
Fishing,848,"7,448,877"
Public Vessel,411,"3,412,168"
Industrial Vessel,100,"3,334,769"
Tanker,824,"2,748,040"
Tug,18,"793,654"
Towing,17,"690,933"


:Layout
   .Bars.I  :Bars   [vessel_type]   (vessels,positions)
   .Bars.II :Bars   [vessel_type]   (positions,vessels)

### The January exclusion, in numbers (Zenodo)

What the report says about January: the rows each dataset lost to the exclusion rule, and the sea-state file that was skipped. The sea-state
lines are one per month file, in the order of the months; the skipped file shows as `ocean_month_matches`, with all the rows of its
November copy dropped (its `rows_in` equals November's). The US dataset has no excluded period; there the cell lists the MMSI classes dropped instead.

In [49]:
if full is not None:
    if DATASET == "zenodo":
        jan = full[full.step.eq("drop_excluded_periods") | full.step.eq("ocean_month_matches")]
        display(jan[["dataset", "step", "rule", "rows_in", "rows_out", "rows_dropped"]].reset_index(drop=True))
        lost = jan[jan.step.eq("drop_excluded_periods")].groupby("dataset").rows_dropped.sum()
        print(f"January 2016 removed: {int(lost.get('AIS', 0)):,} AIS rows, {int(lost.get('weather', 0)):,} weather rows, "
              f"{int(lost.get('ocean', 0)):,} sea-state rows (the edge of a neighbouring month's file); "
              "the sea-state file oc_january.csv is a copy of November and is skipped as a whole.")
    mm = full[full.step.eq("drop_invalid_mmsi") & full.rule.str.startswith("mmsi")].groupby("rule", sort=False)[["rows_in", "rows_dropped"]].sum()
    kept_line = mm.index.str.endswith("kept")
    print(f"\nMMSI classes dropped from the full data (of {int(mm[kept_line].rows_in.sum()):,} rows read in the area, "
          f"{int(mm[kept_line].rows_in.sum() - mm[kept_line].rows_dropped.sum()):,} are ship stations and kept):")
    display(mm[~kept_line].sort_values("rows_in", ascending=False))


MMSI classes dropped from the full data (of 115,556,313 rows read in the area, 115,043,130 are ship stations and kept):


,rows_in,rows_dropped
rule,,
"mmsi: ship, unallocated MID",247187,247187
mmsi: unassigned pattern,101572,101572
mmsi: handheld,78407,78407
mmsi: placeholder,75673,75673
mmsi: craft of a parent ship,9431,9431
"mmsi: craft of a parent ship, unallocated MID",296,296
"mmsi: SAR aircraft, unallocated MID",261,261
mmsi: distress device (SART/MOB/EPIRB),243,243
"mmsi: aid to navigation, unallocated MID",76,76


### What the other notebooks will read

The cell lists the preprocessed folder of the dataset: `sample/preprocessed/` (committed) and `preprocessed/` (after Parts 2 and 3). The notebooks 02 to 05
read through `catraj.datasets` (`load_ais`, `load_weather`, `load_ocean`, `load_coastline`, each with `dataset="zenodo"` or `"us"`).

In [50]:
for label, root in (("sample", OUT), ("full", FULL_OUT)):
    print(f"{label}: {rel(root)}")
    if root.is_dir():
        for p in sorted(root.rglob("*")):
            if p.is_file():
                print(f"{p.stat().st_size / 1e6:9.2f} MB  {p.relative_to(root).as_posix()}")
        a = pd.concat([pd.read_parquet(f, columns=["mmsi", "unix_time"]) for f in spec.ais_outputs(root)])
        print(f"AIS: {len(a):,} positions, {a.mmsi.nunique():,} vessels, {when(a.unix_time.min())} to {when(a.unix_time.max())}\n")
    else:
        print("not built yet: run Parts 2 and 3 with the flags on.\n")

sample: data/UsAIS/sample/preprocessed
    11.69 MB  AIS/ais_2017-11.parquet
     0.47 MB  Coastline/shoreline.parquet
     0.00 MB  preprocessing_report.csv
     0.00 MB  preprocessing_report.txt


AIS: 465,795 positions, 507 vessels, 2017-11-01 00:00:00 to 2017-11-02 23:59:59

full: data/UsAIS/preprocessed
  1315.53 MB  AIS/ais_2017-11.parquet
  1190.86 MB  AIS/ais_2017-12.parquet
    16.38 MB  Coastline/shoreline.parquet
     0.01 MB  preprocessing_report.csv
     0.01 MB  preprocessing_report.txt


AIS: 115,032,399 positions, 13,261 vessels, 2017-11-01 00:00:00 to 2017-12-31 23:59:59

